In [ ]:
import os
os.environ['TWELVE_DATA_API_KEY'] = ''  #
print('✅ API Key مضبوط')

✅ API Key مضبوط


## — Fetcher

In [ ]:
# ============================================================
# fetch_data_usdchf_multitf.py  —  USD/CHF Multi-Timeframe
# ============================================================
# الملفات المحفوظة:
#   USDCHF_5min_raw.csv    ← المصدر الأساسي
#   USDCHF_15min_raw.csv
#   USDCHF_45min_raw.csv
#   USDCHF_1h_raw.csv
#   USDCHF_2h_raw.csv
#   USDCHF_4h_raw.csv
#   USDCHF_1d_raw.csv
#   USDCHF_1W_raw.csv
# ============================================================

import requests
import pandas as pd
import time
import logging
import os
from datetime import datetime, timedelta

# ─── إعدادات ─────────────────────────────────────────────────
TWELVE_DATA_API_KEY = ""
SYMBOL              = "USD/CHF"
SYMBOL_SAFE         = "USDCHF"
OUTPUT_DIR          = "."          # المجلد اللي تحفظ فيه الـ CSVs

# ─── إعدادات كل Timeframe ───────────────────────────────────
# target_candles: أقصى ما يحتاجه الـ training script
# api_interval:   اسم الـ interval في TwelveData API
# years_back:     كم سنة للخلف نريد (حد أقصى واقعي من الـ API)
# delay_sec:      تأخير بين الـ batches (تجنب rate limit)
TIMEFRAME_CONFIG = {
    "5min": {
        "api_interval":  "5min",
        "candle_minutes": 5,
        "target_candles": 630_720,   # 6 سنوات × 252 يوم × 84 شمعة/يوم
        "years_back":    6,
        "delay_sec":     0.4,
        "file":          f"{OUTPUT_DIR}/{SYMBOL_SAFE}_5min_raw.csv",
    },
    "15min": {
        "api_interval":  "15min",
        "candle_minutes": 15,
        "target_candles": 280_320,   # 8 سنوات × 252 × 139 شمعة/يوم
        "years_back":    10,         # ← 10 سنة بدل 8 (margin إضافي)
        "delay_sec":     0.4,
        "file":          f"{OUTPUT_DIR}/{SYMBOL_SAFE}_15min_raw.csv",
    },
    "45min": {
        "api_interval":  "45min",
        "candle_minutes": 45,
        "target_candles": 116_800,   # 10 سنوات
        "years_back":    14,         # ← 14 سنة
        "delay_sec":     0.4,
        "file":          f"{OUTPUT_DIR}/{SYMBOL_SAFE}_45min_raw.csv",
    },
    "1h": {
        "api_interval":  "1h",
        "candle_minutes": 60,
        "target_candles": 105_120,   # 12 سنوات × 252 × 34.7 شمعة/يوم
        "years_back":    12,         # ← 15 سنة (بدل 12 — margin إضافي)
        "delay_sec":     0.4,
        "file":          f"{OUTPUT_DIR}/{SYMBOL_SAFE}_1h_raw.csv",
    },
    "2h": {
        "api_interval":  "2h",
        "candle_minutes": 120,
        "target_candles": 61_320,    # 14 سنوات
        "years_back":    12,         # ← 18 سنة
        "delay_sec":     0.4,
        "file":          f"{OUTPUT_DIR}/{SYMBOL_SAFE}_2h_raw.csv",
    },
    "4h": {
        "api_interval":  "4h",
        "candle_minutes": 240,
        "target_candles": 32_850,    # 15 سنوات
        "years_back":    15,         # ← 20 سنة
        "delay_sec":     0.4,
        "file":          f"{OUTPUT_DIR}/{SYMBOL_SAFE}_4h_raw.csv",
    },
    "1d": {
        "api_interval":  "1day",
        "candle_minutes": 1440,
        "target_candles": 7_560,     # 30 سنوات × 252 شمعة/سنة
        "years_back":    20,         # ← 35 سنة (نجيب أكثر ما يمكن)
        "delay_sec":     0.4,
        "file":          f"{OUTPUT_DIR}/{SYMBOL_SAFE}_1d_raw.csv",
    },
    "1W": {
        "api_interval":  "1week",
        "candle_minutes": 10080,
        "target_candles": 1_560,     # 30 سنوات × 52 أسبوع
        "years_back":    30,
        "delay_sec":     0.4,
        "file":          f"{OUTPUT_DIR}/{SYMBOL_SAFE}_1W_raw.csv",
    },
}

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)


# ══════════════════════════════════════════════════════════════
# TwelveData Fetcher
# ══════════════════════════════════════════════════════════════
class TwelveDataFetcher:
    BASE_URL = "https://api.twelvedata.com/time_series"

    def __init__(self, api_key: str):
        self.api_key = api_key

    def fetch_batch(self, interval: str,
                    start_date: str = None,
                    end_date: str = None,
                    outputsize: int = 5000) -> pd.DataFrame:
        """جلب batch واحد من الـ API."""
        params = {
            'symbol':     SYMBOL,
            'interval':   interval,
            'apikey':     self.api_key,
            'format':     'JSON',
            'order':      'ASC',
            'outputsize': outputsize,
        }
        if start_date:
            params['start_date'] = start_date
        if end_date:
            params['end_date'] = end_date

        try:
            r = requests.get(self.BASE_URL, params=params, timeout=30)
            if r.status_code != 200:
                logging.error(f"HTTP {r.status_code}: {r.text[:200]}")
                return pd.DataFrame()

            data = r.json()
            if 'values' not in data:
                msg = data.get('message', data.get('status', 'unknown error'))
                logging.error(f"API error: {msg}")
                return pd.DataFrame()

            df = pd.DataFrame(data['values'])
            if df.empty:
                return df

            # تأكد من وجود volume (الفوركس أحياناً بدون volume)
            if 'volume' not in df.columns:
                df['volume'] = 1_000_000   # قيمة افتراضية معقولة للفوركس

            for col in ['open', 'high', 'low', 'close', 'volume']:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

            df['datetime'] = pd.to_datetime(df['datetime'])
            df.sort_values('datetime', inplace=True)
            df.reset_index(drop=True, inplace=True)
            return df

        except Exception as e:
            logging.error(f"Fetch exception: {e}")
            return pd.DataFrame()


# ══════════════════════════════════════════════════════════════
# Fetch Single Timeframe
# ══════════════════════════════════════════════════════════════
def fetch_timeframe(tf_name: str, cfg: dict,
                    fetcher: TwelveDataFetcher) -> pd.DataFrame:
    """
    جلب أقصى كمية ممكنة من بيانات timeframe معين.
    يستخدم pagination للخلف (من الأحدث للأقدم).
    """
    interval       = cfg['api_interval']
    target         = cfg['target_candles']
    years_back     = cfg['years_back']
    delay          = cfg['delay_sec']
    candle_minutes = cfg['candle_minutes']

    logging.info(f"\n{'='*55}")
    logging.info(f"جلب {tf_name} | interval={interval} | target={target:,} | {years_back} سنة")
    logging.info(f"{'='*55}")

    # أقدم تاريخ مطلوب
    oldest_wanted = datetime.utcnow() - timedelta(days=365 * years_back)
    end_dt        = datetime.utcnow()

    all_batches  = []
    total_candles = 0
    prev_oldest   = None
    req_count     = 0
    MAX_REQUESTS  = 200   # ceiling لتجنب loops لا نهائية

    while req_count < MAX_REQUESTS:
        # احسب start_date بناءً على حجم الـ batch
        batch_minutes = 5000 * candle_minutes
        start_dt      = end_dt - timedelta(minutes=batch_minutes)

        # لا نذهب قبل أقدم تاريخ مطلوب
        if start_dt < oldest_wanted:
            start_dt = oldest_wanted

        batch = fetcher.fetch_batch(
            interval=interval,
            start_date=start_dt.strftime("%Y-%m-%d %H:%M:%S"),
            end_date=end_dt.strftime("%Y-%m-%d %H:%M:%S"),
            outputsize=5000,
        )

        req_count += 1

        if batch.empty:
            logging.warning(f"  Batch {req_count}: فارغ — توقف")
            break

        all_batches.append(batch)
        total_candles += len(batch)
        oldest_in_batch = batch['datetime'].min()
        newest_in_batch = batch['datetime'].max()

        logging.info(
            f"  Batch {req_count:03d}: {len(batch):,} شمعة | "
            f"إجمالي: {total_candles:,}/{target:,} | "
            f"{oldest_in_batch.date()} → {newest_in_batch.date()}"
        )

        # هل وصلنا للهدف؟
        if total_candles >= target:
            logging.info(f"  ✅ وصلنا للهدف {target:,}")
            break

        # هل وصلنا لأقدم تاريخ مطلوب؟
        if oldest_in_batch <= oldest_wanted:
            logging.info(f"  ✅ وصلنا للحد التاريخي ({years_back} سنة)")
            break

        # هل نفس الـ batch يتكرر؟ (بيانات قديمة غير متوفرة)
        if prev_oldest is not None and oldest_in_batch >= prev_oldest:
            logging.warning(f"  ⚠️ توقف — لا توجد بيانات أقدم من {oldest_in_batch.date()}")
            break

        prev_oldest = oldest_in_batch
        end_dt      = oldest_in_batch - timedelta(seconds=1)

        time.sleep(delay)

    if not all_batches:
        logging.error(f"❌ لا توجد بيانات لـ {tf_name}")
        return pd.DataFrame()

    # دمج وتنظيف
    df = pd.concat(all_batches, ignore_index=True)
    df.drop_duplicates(subset='datetime', keep='first', inplace=True)
    df.sort_values('datetime', inplace=True)
    df.reset_index(drop=True, inplace=True)

    # احتفظ فقط بالعمود المطلوبة
    df = df[['datetime', 'open', 'high', 'low', 'close', 'volume']].copy()

    logging.info(f"  📊 النهائي: {len(df):,} شمعة | "
                 f"{df['datetime'].min().date()} → {df['datetime'].max().date()}")
    return df


# ══════════════════════════════════════════════════════════════
# Update Existing CSV
# ══════════════════════════════════════════════════════════════
def update_existing(csv_path: str, cfg: dict,
                    fetcher: TwelveDataFetcher) -> pd.DataFrame:
    """تحديث ملف CSV موجود بالشموع الجديدة فقط."""
    df_old = pd.read_csv(csv_path)
    df_old['datetime'] = pd.to_datetime(df_old['datetime'])
    last_dt = df_old['datetime'].max()

    logging.info(f"  آخر شمعة موجودة: {last_dt}")

    batch = fetcher.fetch_batch(
        interval=cfg['api_interval'],
        start_date=(last_dt + timedelta(seconds=1)).strftime("%Y-%m-%d %H:%M:%S"),
        end_date=datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
        outputsize=5000,
    )

    if batch.empty:
        logging.info("  ✅ لا توجد شموع جديدة")
        return df_old

    df = pd.concat([df_old, batch], ignore_index=True)
    df.drop_duplicates(subset='datetime', keep='last', inplace=True)
    df.sort_values('datetime', inplace=True)
    df.reset_index(drop=True, inplace=True)

    logging.info(f"  ✅ تحديث: +{len(batch):,} شمعة | إجمالي: {len(df):,}")
    return df


# ══════════════════════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════════════════════
def fetch_all_timeframes(mode: str = 'smart',
                         timeframes: list = None):
    """
    mode:
        'smart'  → يحمّل الملفات الموجودة ويُحدِّثها، يجلب الجديدة من الأول
        'update' → يُحدِّث فقط (يتجاهل الـ TFs التي لا يوجد لها ملف)
        'fresh'  → يُعيد الجلب الكامل لكل شيء

    timeframes: list of TF names to process, e.g. ['1h', '4h', '1d']
                None = جميع الـ timeframes
    """
    fetcher = TwelveDataFetcher(TWELVE_DATA_API_KEY)

    if timeframes is None:
        tfs_to_process = list(TIMEFRAME_CONFIG.keys())
    else:
        tfs_to_process = [tf for tf in timeframes if tf in TIMEFRAME_CONFIG]

    summary = {}

    for tf_name in tfs_to_process:
        cfg      = TIMEFRAME_CONFIG[tf_name]
        csv_path = cfg['file']

        try:
            if mode == 'update':
                if not os.path.exists(csv_path):
                    logging.warning(f"⏭  {tf_name}: ملف غير موجود — تجاوز (mode=update)")
                    summary[tf_name] = 'skipped'
                    continue
                logging.info(f"\n🔄 تحديث {tf_name}...")
                df = update_existing(csv_path, cfg, fetcher)

            elif mode == 'fresh' or not os.path.exists(csv_path):
                if mode != 'fresh' and not os.path.exists(csv_path):
                    logging.info(f"\n⬇️  {tf_name}: ملف غير موجود — جلب كامل...")
                else:
                    logging.info(f"\n🔃 {tf_name}: إعادة جلب كامل (fresh)...")
                df = fetch_timeframe(tf_name, cfg, fetcher)

            else:  # smart: ملف موجود → تحديث
                logging.info(f"\n🔄 {tf_name}: ملف موجود — تحديث بالجديد...")
                df = update_existing(csv_path, cfg, fetcher)

            if df.empty:
                summary[tf_name] = f'❌ فشل'
                continue

            # حفظ
            df.to_csv(csv_path, index=False)
            coverage_pct = len(df) / cfg['target_candles'] * 100
            summary[tf_name] = (
                f"✅ {len(df):,} شمعة "
                f"({coverage_pct:.0f}% من الهدف) | "
                f"{df['datetime'].min().date()} → {df['datetime'].max().date()}"
            )
            logging.info(f"  💾 محفوظ: {csv_path}")

        except Exception as e:
            logging.error(f"❌ خطأ في {tf_name}: {e}", exc_info=True)
            summary[tf_name] = f'❌ خطأ: {e}'

        # راحة قصيرة بين الـ timeframes
        time.sleep(1.0)

    # ملخص نهائي
    print("\n" + "=" * 60)
    print("📊 ملخص النتائج:")
    print("=" * 60)
    for tf, status in summary.items():
        print(f"  {tf:6s}: {status}")
    print("=" * 60)

    return summary


# ══════════════════════════════════════════════════════════════
# Training Script Integration
# ══════════════════════════════════════════════════════════════
def load_raw_data(tf_name: str) -> pd.DataFrame:
    """
    يُستخدم من training script بدل الـ resample.
    يحمّل الـ CSV المناسب مباشرة.
    """
    cfg = TIMEFRAME_CONFIG.get(tf_name)
    if cfg is None:
        raise ValueError(f"Timeframe '{tf_name}' غير معروف")

    csv_path = cfg['file']
    if not os.path.exists(csv_path):
        raise FileNotFoundError(
            f"ملف {csv_path} غير موجود.\n"
            f"شغّل: python fetch_data_eurusd_multitf.py أولاً"
        )

    df = pd.read_csv(csv_path)
    df['datetime'] = pd.to_datetime(df['datetime'])
    df.sort_values('datetime', inplace=True)
    df.reset_index(drop=True, inplace=True)
    logging.info(f"✅ {tf_name}: {len(df):,} شمعة محمّلة من {csv_path}")
    return df


# ══════════════════════════════════════════════════════════════
# تشغيل مباشر
# ══════════════════════════════════════════════════════════════
if __name__ == "__main__":
    import sys

    print("=" * 60)
    print("EUR/USD Multi-Timeframe Data Fetcher")
    print("=" * 60)
    print()
    print("الـ timeframes المتاحة:")
    for tf, cfg in TIMEFRAME_CONFIG.items():
        csv_exists = "✅ موجود" if os.path.exists(cfg['file']) else "❌ غير موجود"
        print(f"  {tf:6s}: {cfg['target_candles']:>8,} شمعة | {cfg['years_back']} سنة | {csv_exists}")
    print()

    # قراءة الـ arguments
    args = sys.argv[1:]

    if '--fresh' in args:
        mode = 'fresh'
    elif '--update' in args:
        mode = 'update'
    else:
        mode = 'smart'

    # تحديد الـ timeframes (كلها أو بعضها)
    tf_filter = [a for a in args if a in TIMEFRAME_CONFIG]
    if not tf_filter:
        tf_filter = None   # كل الـ timeframes

    print(f"Mode: {mode}")
    print(f"Timeframes: {tf_filter or 'الكل'}")
    print()

    # تأكيد المستخدم
    if mode == 'fresh':
        ans = input("⚠️  fresh mode سيحذف ويُعيد جلب كل البيانات. متأكد؟ (y/n): ")
        if ans.lower() != 'y':
            print("تم الإلغاء.")
            exit(0)

    fetch_all_timeframes(mode=mode, timeframes=tf_filter)

    print("\n✅ انتهى! يمكنك الآن تشغيل training script.")
    print()
    print("للتشغيل التالي (تحديث فقط):")
    print("  python fetch_data_eurusd_multitf.py --update")
    print()
    print("لجلب timeframes محددة فقط:")
    print("  python fetch_data_eurusd_multitf.py 1h 4h 1d")
    print("  python fetch_data_eurusd_multitf.py --fresh 1h 4h 1d")

EUR/USD Multi-Timeframe Data Fetcher

الـ timeframes المتاحة:
  5min  :  630,720 شمعة | 6 سنة | ❌ غير موجود
  15min :  280,320 شمعة | 10 سنة | ❌ غير موجود
  45min :  116,800 شمعة | 14 سنة | ❌ غير موجود
  1h    :  105,120 شمعة | 12 سنة | ❌ غير موجود
  2h    :   61,320 شمعة | 12 سنة | ❌ غير موجود
  4h    :   32,850 شمعة | 15 سنة | ❌ غير موجود
  1d    :    7,560 شمعة | 20 سنة | ❌ غير موجود
  1W    :    1,560 شمعة | 30 سنة | ❌ غير موجود

Mode: smart
Timeframes: الكل



/tmp/ipykernel_1321/2754353612.py:187: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  oldest_wanted = datetime.utcnow() - timedelta(days=365 * years_back)
/tmp/ipykernel_1321/2754353612.py:188: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_dt        = datetime.utcnow()
/tmp/ipykernel_1321/2754353612.py:157: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['datetime'] = pd.to_datetime(df['datetime'])
/tmp/ipykernel_1321/2754353612.py:157: UserWarning: Could not infer format, so each element will be parsed individually, falling back to


📊 ملخص النتائج:
  5min  : ✅ 465,755 شمعة (74% من الهدف) | 2020-09-22 → 2026-09-21
  15min : ✅ 171,848 شمعة (61% من الهدف) | 2020-01-29 → 2026-09-21
  45min : ✅ 57,645 شمعة (49% من الهدف) | 2020-01-29 → 2026-09-21
  1h    : ✅ 43,091 شمعة (41% من الهدف) | 2020-01-29 → 2026-09-21
  2h    : ✅ 21,711 شمعة (35% من الهدف) | 2020-01-29 → 2026-09-21
  4h    : ✅ 11,012 شمعة (34% من الهدف) | 2020-01-29 → 2026-09-21
  1d    : ✅ 5,346 شمعة (71% من الهدف) | 2006-09-27 → 2026-09-21
  1W    : ✅ 1,565 شمعة (100% من الهدف) | 1996-09-30 → 2026-09-21

✅ انتهى! يمكنك الآن تشغيل training script.

للتشغيل التالي (تحديث فقط):
  python fetch_data_eurusd_multitf.py --update

لجلب timeframes محددة فقط:
  python fetch_data_eurusd_multitf.py 1h 4h 1d
  python fetch_data_eurusd_multitf.py --fresh 1h 4h 1d


##  — Model Training

In [10]:
# ============================================================
# train_patterns_usdchf_v1.py  —  USD/CHF Pattern Recognition
# ============================================================



import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)
from sklearn.ensemble import (RandomForestClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
import joblib, os, json, logging, warnings
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')
np.random.seed(42)

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    logging.warning("LightGBM not found — stacking will use XGB+RF only")

try:
    import talib as _talib
    HAS_TALIB = True
except ImportError:
    HAS_TALIB = False

# ── Constants ────────────────────────────────────────────────
SYMBOL_SAFE  = "USDCHF"
MODELS_DIR   = "models/usdchf_patterns_v4"
FIB_TOL      = 0.10
SPREAD_BPS   = 1.8  # USD/CHF: سبريد أوسع شوية من EUR
ATR_SL_MULT  = 1.3  # USD/CHF: أهدأ (~0.7x تقلب)
ATR_TP_MULT  = 2.2  # USD/CHF: TP أضيق

# ── Speed Config ─────────────────────────────────────────────
# غيّر SKIP_WEAK_TF=True لو عايز تتجاوز 1W (مش active for live)
SKIP_INACTIVE_TF = False

# ── Timeframe Config v4.1-fast ───────────────────────────────
TIMEFRAME_CONFIG = {
    "5min": {
        # v4.8 ✅ LOCKED — أفضل فريم على الإطلاق
        # v4.7: WR=72.31%, PF=3.74, Consist=100% — لا تلمسه
        "resample_rule": "5min",  "minutes_per_candle": 5,
        "years": 6,               "min_candles": 1000,
        "horizon": 12,            "pattern_window": 30,
        "min_return": 0.0004,     "htf_resample": "1h",
        "zigzag_pct": 0.002,      "pattern_max_age": 25,
        "confirmation_bars": 2,   "higher_tf": "1h",
        "min_pattern_quality": 0.20,
        "buy_threshold": 0.58,    "sell_threshold": 0.40,
        "fold_buy_threshold": 0.56, "fold_sell_threshold": 0.44,
        "confidence_threshold": 0.58,
        "atr_sl_mult": 1.3,       "atr_tp_mult": 2.2,
        "adx_min": 14,            "active_for_live": True,
        "n_splits": 6,
        "use_ensemble": False,    "use_feature_selection": False,
        "use_calibration": False,
        "use_median_split": False, "use_directional": False,
        "spw_cap": 5.0,
        "require_htf_align": True, "htf_override_threshold": 0.72,
    },

    "15min": {
        # v4.8 ✅ RESTORED — نرجع لـ v4.5 الأفضل
        # v4.5: WR=51.38%, PF=2.19, Consist=100%
        # v4.7 أضرّ بسبب horizon=7 → نرجع لـ horizon=8
        "resample_rule": "15min", "minutes_per_candle": 15,
        "years": 6,               "min_candles": 600,
        "horizon": 8,             "pattern_window": 20,
        "min_return": 0.0004,     "htf_resample": "1h",
        "zigzag_pct": 0.0015,     "pattern_max_age": 20,
        "confirmation_bars": 2,   "higher_tf": "1h",
        "min_pattern_quality": 0.12,
        "buy_threshold": 0.57,    "sell_threshold": 0.41,
        "fold_buy_threshold": 0.55, "fold_sell_threshold": 0.45,
        "confidence_threshold": 0.56,
        "atr_sl_mult": 1.1,       "atr_tp_mult": 2.4,
        "adx_min": 12,            "active_for_live": True,
        "n_splits": 6,
        "use_ensemble": False,    "use_feature_selection": False,
        "use_calibration": False,
        "use_median_split": False, "use_directional": False,
        "spw_cap": 5.0,
        "require_htf_align": False, "htf_override_threshold": 0.70,
    },



    # ─────────────────────────────────────────────────────────
    # v4.3 FIX — الإصلاح الجذري للفريمات الضعيفة
    # المشكلة: Target Imbalance شديد → الموديل يقول 0 دائماً
    #          SPW cap=5.0 لم يكن كافياً
    #          Calibration يضغط probabilities تحت الـ threshold
    #          Threshold لا يزال مرتفعاً
    # الحل:    median-split target + رفع spw_cap + إلغاء calibration
    #          + تخفيض thresholds بشكل حاد + رفع horizon
    # ─────────────────────────────────────────────────────────
    "45min": {
        # v4.8 ✅ RESTORED — نرجع لـ v4.5 الأفضل
        # v4.5: WR=47.96%, PF=1.90, Consist=100%, Sharpe=0.16
        # v4.7 أضرّ بسبب SL=0.9 → نرجع لـ SL=1.0 مع directional
        "resample_rule": "45min", "minutes_per_candle": 45,
        "years": 10,              "min_candles": 400,   # ★ V4: 12→10 (توازن)
        "horizon": 7,             "pattern_window": 14,  # ★ V4: 8→7, window 12→14 (أنماط أنضج)
        "min_return": 0.0003,     "htf_resample": "1h",
        "zigzag_pct": 0.0013,     "pattern_max_age": 20,  # ★ V4: 0.0010→0.0013 (أنماط أقوى مش أكتر)
        "confirmation_bars": 2,   "higher_tf": "1h",
        "min_pattern_quality": 0.05,
        "buy_threshold": 0.58,    "sell_threshold": 0.40,   # ★ V4: gap 0.16→0.20 (تشديد — كان توسيع ضرّه)
        "fold_buy_threshold": 0.55, "fold_sell_threshold": 0.43,
        "confidence_threshold": 0.55,
        "atr_sl_mult": 1.0,       "atr_tp_mult": 2.3,
        "adx_min": 14,             "active_for_live": True,  # ★ V4: 10→14 (فلترة أقوى)
        "n_splits": 5,
        "use_ensemble": False,
        "use_feature_selection": True,
        "use_calibration": False,
        "use_median_split": False,
        "use_directional": True,
        "spw_cap": 6.0,
        "require_htf_align": True, "htf_override_threshold": 0.62,
    },




    "1h": {
        # ★ V2: رفع AUC (كان 0.522 عشوائي) — سنوات أكتر + أنماط أكتر + HTF
        "resample_rule": "1h",    "minutes_per_candle": 60,
        "years": 10,              "min_candles": 400,   # ★ V2: 6→10 (أنماط أكتر للتعلم)
        "horizon": 6,             "pattern_window": 12,  # ★ V2: 9→6 (حسم أوضح), window 15→12
        "min_return": 0.0003,     "htf_resample": "4h",
        "zigzag_pct": 0.0010,     "pattern_max_age": 20,  # ★ V2: 0.0015→0.0010 (يكتشف أنماط أكتر)
        "confirmation_bars": 2,   "higher_tf": "4h",
        "min_pattern_quality": 0.03,                       # ★ V2: 0.05→0.03 (عينة أكبر)
        "buy_threshold": 0.58,    "sell_threshold": 0.42,
        "fold_buy_threshold": 0.55, "fold_sell_threshold": 0.45,
        "confidence_threshold": 0.53,
        "atr_sl_mult": 1.0,       "atr_tp_mult": 2.6,
        "adx_min": 14,             "active_for_live": True,  # ★ V2: 12→14 (جودة أعلى)
        "n_splits": 5,
        "use_ensemble": False,
        "use_feature_selection": True,
        "use_calibration": False,
        "use_median_split": False,
        "use_directional": True,
        "spw_cap": 5.0,
        "require_htf_align": True, "htf_override_threshold": 0.62,  # ★ V2: فعّل HTF alignment
    },




    "2h": {
        "resample_rule": "2h",
        "minutes_per_candle": 120,
        "years": 10,                     # ★ V2: 6→10 (أنماط أكتر)
        "min_candles": 250,
        "horizon": 6,                    # ★ V2: 7→6 (حسم أوضح)
        "pattern_window": 12,
        "min_return": 0.00035,
        "htf_resample": "1d",
        "zigzag_pct": 0.0010,            # ★ V2: 0.0015→0.0010 (أنماط أكتر)
        "pattern_max_age": 20,
        "confirmation_bars": 2,
        "higher_tf": "1d",

        "min_pattern_quality": 0.03,      # ★ V2: عينة أكبر للتعلم
        "buy_threshold": 0.58,            # ★ CHF: gap 0.16 (أوسع)
        "sell_threshold": 0.42,
        "fold_buy_threshold": 0.53,
        "fold_sell_threshold": 0.45,
        "confidence_threshold": 0.555,

        "atr_sl_mult": 1.0,               # ★ CHF
        "atr_tp_mult": 2.8,               # TP أعلى → RR أفضل

        "adx_min": 8,                     # ← ارفع (اتجاه أقوى)
        "active_for_live": True,
        "n_splits": 5,
        "use_ensemble": False,
        "use_feature_selection": True,
        "use_calibration": False,
        "use_median_split": False,
        "use_directional": True,          # مهم
        "spw_cap": 5.0,
        "require_htf_align": True,        # ← فعّل HTF
        "htf_override_threshold": 0.62,


    },




    "4h": {
        # v4.8 ✅ IMPROVED — v4.7 أعطى PF=2.02 Consist=100%
        # نحسّن WR=38.3%→42%+ بتضييق SL قليلاً
        "resample_rule": "4h",    "minutes_per_candle": 240,
        "years": 14,              "min_candles": 200,   # ★ V2: 6→14 (4h محتاج بيانات كتير — كان AUC=0.513)
        "horizon": 8,             "pattern_window": 12,  # ★ V2: 6→8 (أنماط 4h تحتاج وقت للحسم)
        "min_return": 0.0003,     "htf_resample": "1d",
        "zigzag_pct": 0.0010,     "pattern_max_age": 25,  # ★ V2: أنماط أكتر
        "confirmation_bars": 2,   "higher_tf": "1d",
        "min_pattern_quality": 0.015,                      # ★ V2: عينة أكبر
        "buy_threshold": 0.57,    "sell_threshold": 0.41,
        "fold_buy_threshold": 0.55, "fold_sell_threshold": 0.45,
        "confidence_threshold": 0.58,
        "atr_sl_mult": 1.1,       "atr_tp_mult": 2.6,
        "adx_min": 12,             "active_for_live": True,
        "n_splits": 4,
        "use_ensemble": False,
        "use_feature_selection": True,
        "use_calibration": False,
        "use_median_split": False,
        "use_directional": True,
        "spw_cap": 5.0,
        "require_htf_align": True, "htf_override_threshold": 0.62,
    },




    "1d": {
        # v4.8 ⚠️ BEST ATTEMPT — أفضل ما وصلنا: v4.5 PF=1.17
        # المشكلة: بيانات 5,879 شمعة → trades قليلة
        # الحل: median_split + أقل فلاتر + SL/TP متوازن
        "resample_rule": "1B",    "minutes_per_candle": 1440,
        "years": 25,              "min_candles": 100,
        "horizon": 10,            "pattern_window": 8,   # ★ V3: 8→10 (حسم أوضح), window 10→8
        "min_return": 0.0003,     "htf_resample": "W-FRI",
        "zigzag_pct": 0.0006,     "pattern_max_age": 30,  # ★ V3: 0.0008→0.0006 (أنماط أكتر بكتير)
        "confirmation_bars": 1,   "higher_tf": "1W",
        "min_pattern_quality": 0.006,
        "buy_threshold": 0.53,    "sell_threshold": 0.44,
        "fold_buy_threshold": 0.51, "fold_sell_threshold": 0.49,
        "confidence_threshold": 0.52,
        "atr_sl_mult": .9,       "atr_tp_mult": 2.6,   # RR=2.92 لتعويض WR المنخفض
        "adx_min": 6,             "active_for_live": True,
        "n_splits": 3,             # ★ V3: 4→3 (folds أكبر جداً = ثبات أعلى — يحل Consist 50%)
        "use_ensemble": False,
        "use_feature_selection": True,
        "use_calibration": False,
        "use_median_split": True,  # median يعطي أكثر balance
        "use_directional": False,  # directional حذف 80% من البيانات
        "spw_cap": 3.0,
        "require_htf_align": False, "htf_override_threshold": 0.55,
    },




    "1W": {
        # v4.8 ⛔ DISABLED — Overfitting (PF=35 من 4 صفقات)
        # يُستخدم كـ HTF context فقط للـ 1d
        # سيُفعَّل لاحقاً بعد جمع بيانات أكثر
        "resample_rule": "W-FRI", "minutes_per_candle": 10080,
        "years": 30,              "min_candles": 40,    # ★ V3: 20→30 سنة، min 60→40
        "horizon": 5,             "pattern_window": 5,   # ★ V3: window 6→5 (أنماط أصغر)
        "min_return": 0.0004,     "htf_resample": None,
        "zigzag_pct": 0.0005,     "pattern_max_age": 25,  # ★ V3: 0.001→0.0005 (أقصى كشف)
        "confirmation_bars": 1,   "higher_tf": None,
        "min_pattern_quality": 0.004,
        "buy_threshold": 0.53,    "sell_threshold": 0.45,
        "fold_buy_threshold": 0.51, "fold_sell_threshold": 0.49,
        "confidence_threshold": 0.52,
        "atr_sl_mult": 1.0,       "atr_tp_mult": 2.8,
        "adx_min": 3,             "active_for_live": True,   # ★ V3: أعدنا تفعيله لاختبار التخفيف الأقصى
        "n_splits": 3,
        "use_ensemble": False,
        "use_feature_selection": True,
        "use_calibration": False,
        "use_median_split": True,
        "use_directional": False,
        "spw_cap": 3.0,
        "require_htf_align": False, "htf_override_threshold": 0.55,
    },
}

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("training_usdchf_v4.log", encoding='utf-8'),
        logging.StreamHandler()
    ]
)


# ══════════════════════════════════════════════════════════════
# ZIGZAG NON-REPAINTING
# ══════════════════════════════════════════════════════════════
def zigzag_swings(high, low, pct_thresh=0.01, confirmation_bars=2):
    highs = high.values if hasattr(high, 'values') else np.asarray(high)
    lows  = low.values  if hasattr(low,  'values') else np.asarray(low)
    n = len(highs)
    if n < confirmation_bars + 3:
        return np.array([], dtype=int), np.array([], dtype=int)
    peak_idx, trough_idx = [], []
    direction = 1; last_high = 0; last_low = 0
    confirmed_h = -1; confirmed_l = -1
    for i in range(1, n):
        if direction == 1:
            if highs[i] > highs[last_high]:
                last_high = i
            if (lows[i] < highs[last_high] * (1.0 - pct_thresh)
                    and i - last_high >= confirmation_bars):
                if last_high > confirmed_h:
                    peak_idx.append(last_high)
                    confirmed_h = last_high
                direction = -1; last_low = i
        else:
            if lows[i] < lows[last_low]:
                last_low = i
            if (highs[i] > lows[last_low] * (1.0 + pct_thresh)
                    and i - last_low >= confirmation_bars):
                if last_low > confirmed_l:
                    trough_idx.append(last_low)
                    confirmed_l = last_low
                direction = 1; last_high = i
    return np.array(peak_idx, dtype=int), np.array(trough_idx, dtype=int)


def leg_ratio(a, b):
    return abs(b) / abs(a) if abs(a) > 1e-9 else 0.0


# ══════════════════════════════════════════════════════════════
# FEATURE ENGINEER v4.1-fast
# ══════════════════════════════════════════════════════════════
class PatternFeatureEngineerV4:
    def __init__(self):
        self.scaler          = RobustScaler()
        self.feature_columns = None
        self.scaler_fitted   = False
        self.selector        = None
        self.selected_cols   = None

    # ─── Base Features (مُحسَّنة السرعة) ───────────────────
    def add_base_features(self, df):
        df = df.copy()
        c, h, l, o, v = (df['close'], df['high'], df['low'],
                         df['open'], df['volume'])
        n = len(df)

        p5   = min(5,   max(2, n // 20))
        p14  = min(14,  max(3, n // 8))
        p20  = min(20,  max(4, n // 6))
        p50  = min(50,  max(6, n // 4))
        p100 = min(100, max(8, n // 3))
        p200 = min(200, max(10, n // 2))

        # EMAs
        df['ema_5']   = c.ewm(span=p5,   adjust=False).mean()
        df['ema_20']  = c.ewm(span=p20,  adjust=False).mean()
        df['ema_50']  = c.ewm(span=p50,  adjust=False).mean()
        df['ema_100'] = c.ewm(span=p100, adjust=False).mean()
        df['ema_200'] = c.ewm(span=p200, adjust=False).mean()
        df['trend']   = np.sign(df['ema_20'] - df['ema_50'])
        df['trend_20_100'] = np.sign(df['ema_20'] - df['ema_100'])

        # ATR
        c_s = c.shift(1)
        tr = pd.concat([h - l, (h - c_s).abs(), (l - c_s).abs()], axis=1).max(axis=1)
        df['atr']     = tr.rolling(p14).mean()
        df['atr_pct'] = df['atr'] / (c + 1e-9)
        atr_ma = df['atr'].rolling(p50).mean()
        df['atr_normalized'] = df['atr'] / (atr_ma + 1e-9)

        # RSI
        delta = c.diff()
        ag = delta.clip(lower=0).rolling(p14).mean()
        al = (-delta.clip(upper=0)).rolling(p14).mean().clip(lower=1e-9)
        df['rsi'] = 100 - (100 / (1 + ag / al))
        df['rsi_slope'] = df['rsi'].diff(3)
        rsi_roll = df['rsi'].rolling(p14)
        df['stoch_rsi'] = (df['rsi'] - rsi_roll.min()) / (rsi_roll.max() - rsi_roll.min() + 1e-9)

        # MACD
        ema12 = c.ewm(span=min(12, max(3, n//8)), adjust=False).mean()
        ema26 = c.ewm(span=min(26, max(5, n//5)), adjust=False).mean()
        df['macd']      = ema12 - ema26
        df['macd_sig']  = df['macd'].ewm(span=min(9, max(3, n//10)), adjust=False).mean()
        df['macd_hist'] = df['macd'] - df['macd_sig']
        df['macd_cross'] = np.sign(df['macd_hist']).diff()

        # Bollinger Bands
        sma20 = c.rolling(p20).mean()
        std20 = c.rolling(p20).std()
        df['bb_upper'] = sma20 + 2 * std20
        df['bb_lower'] = sma20 - 2 * std20
        df['bb_mid']   = sma20
        df['bb_pctb']  = (c - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'] + 1e-9)
        df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / (sma20 + 1e-9)
        # ← حذف bb_squeeze (rolling quantile ثقيلة جداً)
        # بديل خفيف: مقارنة بمتوسط bb_width
        df['bb_squeeze'] = (df['bb_width'] < df['bb_width'].rolling(p50).mean() * 0.75).astype(float)

        # MFI
        typ_p  = (h + l + c) / 3
        raw_mf = typ_p * v
        mf_dir = np.where(typ_p > typ_p.shift(1), 1, -1)
        pos_mf = (raw_mf * (mf_dir > 0)).rolling(p14).sum()
        neg_mf = (raw_mf * (mf_dir < 0)).rolling(p14).sum()
        df['mfi'] = 100 - (100 / (1 + pos_mf / (neg_mf + 1e-9)))

        # CCI
        cci_window = min(20, max(5, n // 6))
        mean_tp    = typ_p.rolling(cci_window).mean()
        mean_dev   = (typ_p - mean_tp).abs().rolling(cci_window).mean()
        df['cci']  = (typ_p - mean_tp) / (0.015 * mean_dev + 1e-9)

        # Volume
        vol_ma = v.rolling(p20).mean().shift(1).fillna(v.rolling(p20).mean())
        df['vol_ratio'] = v / (vol_ma + 1e-9)
        df['vol_surge'] = (df['vol_ratio'] > 2.0).astype(float)
        df['vol_trend'] = v.rolling(p5).mean() / (v.rolling(p20).mean() + 1e-9)

        # VWAP
        vwap_w = min(200, max(5, n // 3))
        df['vwap']       = ((typ_p * v).rolling(vwap_w, min_periods=1).sum()
                           / (v.rolling(vwap_w, min_periods=1).sum() + 1e-9))
        df['dist_vwap']  = (c - df['vwap']) / (df['atr'] + 1e-9)
        df['above_vwap'] = (c > df['vwap']).astype(float)

        # OBV
        obv = (np.sign(c.diff()) * v).fillna(0).cumsum()
        s5  = min(5, max(1, n // 15))
        df['obv_slope']  = obv.diff(s5) / (obv.shift(s5).abs() + 1e-9)
        df['obv_ma']     = obv.rolling(p20).mean()
        df['obv_signal'] = np.sign(obv - df['obv_ma'])

        # Returns & Volatility
        df['returns']    = c.pct_change()
        df['returns_5']  = c.pct_change(5)
        df['returns_10'] = c.pct_change(10)
        df['volatility'] = df['returns'].rolling(p20).std()
        df['vol_of_vol'] = df['volatility'].rolling(p20).std()

        # Candle
        df['body_pct']     = (c - o).abs() / (h - l + 1e-9)
        df['upper_wick']   = (h - c.clip(upper=o)) / (h - l + 1e-9)
        df['lower_wick']   = (c.clip(lower=o) - l) / (h - l + 1e-9)
        df['body_size_ma'] = df['body_pct'].rolling(p20).mean()
        df['wick_ratio']   = (df['upper_wick'] + df['lower_wick']) / (df['body_pct'] + 1e-9)
        df['is_bullish_candle']   = (c > o).astype(float)
        df['candle_range_ratio']  = (h - l) / (df['atr'] + 1e-9)

        # Price Position
        sr_w   = min(p50, max(5, n // 4))
        high_n = h.rolling(p50).max()
        low_n  = l.rolling(p50).min()
        df['price_position'] = (c - low_n) / (high_n - low_n + 1e-9)
        df['near_high_50']   = (c > high_n * 0.98).astype(float)
        df['near_low_50']    = (c < low_n * 1.02).astype(float)

        # Time Features
        df['hour']      = df['datetime'].dt.hour
        df['dow']       = df['datetime'].dt.dayofweek
        df['month']     = df['datetime'].dt.month
        df['hour_sin']  = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos']  = np.cos(2 * np.pi * df['hour'] / 24)
        df['dow_sin']   = np.sin(2 * np.pi * df['dow'] / 5)
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        # ★ USD/CHF (Swissy) sessions — أوروبا (زيورخ) + نيويورك + SNB
        df['is_asian']   = ((df['hour'] >= 0) & (df['hour'] < 7)).astype(float)   # هادئة على Swissy
        df['is_european'] = ((df['hour'] >= 7) & (df['hour'] < 16)).astype(float) # ★ CHF home (زيورخ)
        df['is_london']  = ((df['hour'] >= 7) & (df['hour'] <= 16)).astype(float) # alias أوروبا
        df['is_ny']      = ((df['hour'] >= 13) & (df['hour'] <= 21)).astype(float)
        df['is_overlap'] = ((df['is_european'] == 1) & (df['is_ny'] == 1)).astype(float)
        # ─── فيتشورز خاصة بـ USD/CHF ───
        df['is_swiss_open']    = (df['hour'] == 7).astype(float)                   # افتتاح زيورخ
        df['is_swiss_data']    = ((df['hour'] >= 6) & (df['hour'] < 8)).astype(float)  # Swiss CPI/KOF
        df['is_snb_window']    = ((df['hour'] >= 7) & (df['hour'] < 9)).astype(float)  # SNB (07:30 UTC)
        df['is_us_data']       = ((df['hour'] >= 12) & (df['hour'] < 14)).astype(float) # US NFP/CPI/Fed
        df['is_snb_day']       = (df['dow'] == 3).astype(float)                    # SNB عادة الخميس
        df['is_swissy_risky']  = (
            (df['hour'] == 7) | ((df['hour'] >= 6) & (df['hour'] < 9)) | ((df['hour'] >= 12) & (df['hour'] < 14))
        ).astype(float)

        # S/R
        res = h.shift(1).rolling(sr_w, min_periods=1).max()
        sup = l.shift(1).rolling(sr_w, min_periods=1).min()
        df['dist_to_resistance'] = (res - c) / (df['atr'] + 1e-9)
        df['dist_to_support']    = (c - sup) / (df['atr'] + 1e-9)
        df['near_breakout']      = (
            (df['dist_to_resistance'] < 0.5) | (df['dist_to_support'] < 0.5)
        ).astype(float)
        df['sr_position'] = (c - sup) / (res - sup + 1e-9)

        # Volatility Regime (خفيف — rank بدل quantile)
        vol_rank = df['atr'].rolling(min(100, max(10, n//3)), min_periods=5).rank(pct=True)
        df['vol_regime'] = np.select([vol_rank < 0.33, vol_rank > 0.66], [0, 2], default=1)

        # Momentum Score
        df['momentum_score'] = (
            (df['rsi'] - 50) / 50
            + df['macd_hist'].clip(-0.01, 0.01) / 0.01
            + (df['bb_pctb'] - 0.5)
        ) / 3

        # ── v4 Extra Features ────────────────────────────────
        # Keltner Channel
        kc_ema = c.ewm(span=p20, adjust=False).mean()
        kc_rng = df['kc_upper'] = kc_ema + 2 * df['atr']
        df['kc_lower'] = kc_ema - 2 * df['atr']
        df['kc_pctk']  = (c - df['kc_lower']) / (df['kc_upper'] - df['kc_lower'] + 1e-9)
        df['squeeze_momentum'] = (df['bb_width'] < (df['kc_upper'] - df['kc_lower']) / (kc_ema + 1e-9)).astype(float)

        # Williams %R & Stochastic (يشتركان في نفس rolling)
        hw = h.rolling(p14).max()
        lw = l.rolling(p14).min()
        df['williams_r'] = (hw - c) / (hw - lw + 1e-9) * -100
        df['stoch_k']    = (c - lw) / (hw - lw + 1e-9) * 100
        df['stoch_d']    = df['stoch_k'].rolling(3).mean()

        # Trend Consistency
        df['trend_consistency'] = (df['ema_5'] > df['ema_20']).rolling(p14).mean()

        # Multi-TF Momentum
        df['mom_3']  = c.pct_change(3)
        df['mom_5']  = c.pct_change(5)
        df['mom_10'] = c.pct_change(10)
        df['mom_divergence'] = df['mom_5'] - df['mom_10']

        # Candle Sequence
        bull_mask = (c > o).astype(float)
        df['bull_sequence'] = bull_mask.rolling(5).sum() / 5
        df['bear_sequence'] = (1 - bull_mask).rolling(5).sum() / 5

        # RR potential
        df['rr_potential'] = df['atr'] * 3.0 / (df['atr'] * 1.5 + 1e-9)

        return df

    # ─── HTF Trend ────────────────────────────────────────────
    def add_htf_trend(self, df, htf_resample=None):
        df = df.copy()
        htf_cols = ['htf_trend', 'htf_ema_slope', 'htf_rsi',
                    'htf_above_ema200', 'htf_volatility', 'htf_dist_ema20',
                    'htf_momentum', 'htf_vol_regime']
        for col in htf_cols:
            df[col] = 0.0
        if htf_resample is None:
            return df
        try:
            df_s = df.sort_values('datetime').reset_index(drop=True)
            df_s['datetime'] = pd.to_datetime(df_s['datetime'])
            htf = (df_s.set_index('datetime')
                   .resample(htf_resample)
                   .agg({'open': 'first', 'high': 'max',
                         'low': 'min', 'close': 'last', 'volume': 'sum'})
                   .dropna().reset_index())
            if 'datetime' not in htf.columns:
                htf = htf.rename(columns={htf.columns[0]: 'datetime'})
            htf['datetime'] = pd.to_datetime(htf['datetime'])
            if len(htf) < 5:
                return df
            c_h = htf['close']; nh = len(htf)
            sp20  = min(20,  max(4, nh // 5))
            sp50  = min(50,  max(5, nh // 4))
            sp200 = min(200, max(10, nh // 2))
            htf['e20']  = c_h.ewm(span=sp20,  adjust=False).mean()
            htf['e50']  = c_h.ewm(span=sp50,  adjust=False).mean()
            htf['e200'] = c_h.ewm(span=sp200, adjust=False).mean()
            htf['htf_trend']        = np.sign(htf['e50'] - htf['e200'])
            htf['htf_ema_slope']    = htf['e50'].diff(min(3, max(1, nh//15))) / (htf['e50'].shift(3) + 1e-9)
            htf['htf_above_ema200'] = (c_h > htf['e200']).astype(float)
            htf['htf_dist_ema20']   = (c_h - htf['e20']) / (c_h.rolling(sp20).std() + 1e-9)
            htf['htf_volatility']   = c_h.pct_change().rolling(sp20).std()
            htf['htf_momentum']     = c_h.pct_change(min(5, nh//10))
            atr_rank_h = (htf['high'] - htf['low']).rolling(min(50, nh//3), min_periods=3).rank(pct=True)
            htf['htf_vol_regime']   = np.select([atr_rank_h < 0.33, atr_rank_h > 0.66], [0, 2], default=1)
            p14h = min(14, max(3, nh//6))
            d = c_h.diff()
            g = d.clip(lower=0).rolling(p14h).mean()
            ls = (-d.clip(upper=0)).rolling(p14h).mean().clip(lower=1e-9)
            htf['htf_rsi'] = 100 - (100 / (1 + g / ls))
            slim = htf[['datetime'] + htf_cols].sort_values('datetime').reset_index(drop=True)
            merged = pd.merge_asof(df_s, slim, on='datetime', direction='backward')
            for col in htf_cols:
                if col in merged.columns:
                    df_s[col] = merged[col].fillna(0).values
            for col in htf_cols:
                df[col] = df_s[col].fillna(0).values
        except Exception as e:
            logging.warning(f"HTF error: {e}")
        return df

    # ─── ADX / Regime ─────────────────────────────────────────
    def add_regime_features(self, df):
        df = df.copy()
        n = len(df)
        if HAS_TALIB:
            try:
                df['adx'] = _talib.ADX(
                    df['high'].values, df['low'].values,
                    df['close'].values, timeperiod=14)
            except Exception:
                df['adx'] = (df['atr'] / (df['close'] + 1e-9) * 500).clip(0, 100)
        else:
            hi, lo, cl = df['high'], df['low'], df['close']
            up   = hi.diff()
            down = -lo.diff()
            pdm  = np.where((up > down) & (up > 0), up, 0.0)
            ndm  = np.where((down > up) & (down > 0), down, 0.0)
            tr   = pd.concat([(hi-lo), (hi-cl.shift()).abs(), (lo-cl.shift()).abs()], axis=1).max(axis=1)
            p14  = min(14, max(3, n // 6))
            atr14  = tr.ewm(span=p14, adjust=False).mean()
            pdi14  = pd.Series(pdm).ewm(span=p14, adjust=False).mean() / (atr14 + 1e-9) * 100
            ndi14  = pd.Series(ndm).ewm(span=p14, adjust=False).mean() / (atr14 + 1e-9) * 100
            dx     = ((pdi14 - ndi14).abs() / (pdi14 + ndi14 + 1e-9) * 100)
            df['adx']  = dx.ewm(span=p14, adjust=False).mean()
            df['pdi']  = pdi14
            df['ndi']  = ndi14
            df['di_diff'] = pdi14 - ndi14

        vol_win = min(100, max(10, n // 3))
        atr_rank = df['atr'].rolling(vol_win, min_periods=5).rank(pct=True)
        df['volatility_regime'] = np.select([atr_rank < 0.33, atr_rank > 0.66], [0, 2], default=1)
        df['trend_strength']    = df['adx'] / 50.0
        df['market_regime']     = (df['adx'] > 20).astype(float)
        df['trending_market']   = (df['adx'] > 25).astype(float)
        df['ranging_market']    = (df['adx'] < 20).astype(float)
        return df

    # ─── Order Flow ───────────────────────────────────────────
    def add_order_flow(self, df):
        df = df.copy()
        n  = len(df)
        cmf_win = min(21, max(5, n // 8))
        mfv = (((df['close'] - df['low']) - (df['high'] - df['close']))
               / (df['high'] - df['low'] + 1e-9) * df['volume'])
        df['cmf']        = mfv.rolling(cmf_win).sum() / (df['volume'].rolling(cmf_win).sum() + 1e-9)
        df['cmf_signal'] = np.sign(df['cmf'])
        df['buy_pressure']  = (df['close'] - df['low']) / (df['high'] - df['low'] + 1e-9)
        df['sell_pressure'] = (df['high'] - df['close']) / (df['high'] - df['low'] + 1e-9)
        df['pressure_diff'] = df['buy_pressure'] - df['sell_pressure']
        return df

    # ─── Classic Patterns ─────────────────────────────────────
    def detect_classic_patterns(self, df, window=20, zigzag_pct=0.01,
                                 max_age=20, confirmation_bars=2):
        df = df.copy()
        n  = len(df)
        hi = df['high'].values
        lo = df['low'].values
        cl = df['close'].values
        vl = df['volume'].values

        cols = [
            'hs_pattern', 'hs_direction', 'hs_strength', 'hs_valid',
            'double_pattern', 'double_direction', 'double_strength', 'double_valid',
            'triple_pattern', 'triple_direction', 'triple_strength', 'triple_valid',
            'triangle_pattern', 'triangle_direction', 'triangle_strength',
            'flag_pattern', 'flag_direction', 'flag_strength',
            'wedge_pattern', 'wedge_direction', 'wedge_strength',
            'channel_pattern', 'channel_direction', 'channel_strength',
            'cup_handle_pattern', 'cup_handle_strength', 'cup_handle_valid',
            'rectangle_pattern', 'rectangle_direction', 'rectangle_strength',
            'pattern_age', 'pattern_volume_confirm', 'pattern_intensity'
        ]
        for col in cols:
            df[col] = 0.0

        vol_s  = pd.Series(vl)
        vol_ma = (vol_s.rolling(20, min_periods=3).mean()
                  .shift(1).fillna(vol_s.rolling(20, min_periods=3).mean()).values)
        atr_v  = df['atr'].values

        hi_idx, lo_idx = zigzag_swings(
            df['high'], df['low'], pct_thresh=zigzag_pct,
            confirmation_bars=confirmation_bars)
        swings = sorted(
            [{'idx': i, 'price': hi[i], 'type': 'H'} for i in hi_idx]
            + [{'idx': i, 'price': lo[i], 'type': 'L'} for i in lo_idx],
            key=lambda x: x['idx'])

        for si in range(2, len(swings)):
            pts    = swings[:si + 1]
            i      = pts[-1]['idx']
            if i >= n or i + 1 >= n:
                continue
            a      = max(atr_v[i], 1e-9)
            types  = [p['type'] for p in pts]
            prices = [p['price'] for p in pts]
            vc     = float(vl[i] / (vol_ma[i] + 1e-9) > 1.2)

            if len(pts) >= 5:
                t5 = types[-5:]; p5 = prices[-5:]
                if t5 == ['H', 'L', 'H', 'L', 'H']:
                    sh_l, nk_l, head, nk_r, sh_r = p5
                    neck = (nk_l + nk_r) / 2
                    if head > sh_l * 1.002 and head > sh_r * 1.002:
                        s = min(1.0, (head - max(sh_l, sh_r)) / (a * 2)) * (1.3 if vc else 0.9)
                        df.at[i, 'hs_pattern']   = 1.0
                        df.at[i, 'hs_direction'] = -1.0
                        df.at[i, 'hs_strength']  = min(1.0, s)
                        df.at[i, 'hs_valid']     = float(cl[i] < neck)
                elif t5 == ['L', 'H', 'L', 'H', 'L']:
                    sh_l, nk_l, head, nk_r, sh_r = p5
                    neck = (nk_l + nk_r) / 2
                    if head < sh_l * 0.998 and head < sh_r * 0.998:
                        s = min(1.0, (min(sh_l, sh_r) - head) / (a * 2)) * (1.3 if vc else 0.9)
                        df.at[i, 'hs_pattern']   = 1.0
                        df.at[i, 'hs_direction'] = 1.0
                        df.at[i, 'hs_strength']  = min(1.0, s)
                        df.at[i, 'hs_valid']     = float(cl[i] > neck)

            if len(pts) >= 3:
                t3 = types[-3:]; p3 = prices[-3:]
                if t3 == ['H', 'L', 'H']:
                    h1, valley, h2 = p3
                    if abs(h1 - h2) / a < 2.0:
                        s = (1.0 - abs(h1 - h2) / (a * 2.0 + 1e-9)) * (1.2 if vc else 0.9)
                        df.at[i, 'double_pattern']   = 1.0
                        df.at[i, 'double_direction'] = -1.0
                        df.at[i, 'double_strength']  = min(1.0, s)
                        df.at[i, 'double_valid']     = float(cl[i] < valley)
                elif t3 == ['L', 'H', 'L']:
                    l1, peak, l2 = p3
                    if abs(l1 - l2) / a < 2.0:
                        s = (1.0 - abs(l1 - l2) / (a * 2.0 + 1e-9)) * (1.2 if vc else 0.9)
                        df.at[i, 'double_pattern']   = 1.0
                        df.at[i, 'double_direction'] = 1.0
                        df.at[i, 'double_strength']  = min(1.0, s)
                        df.at[i, 'double_valid']     = float(cl[i] > peak)

            if len(pts) >= 5:
                t5 = types[-5:]; p5 = prices[-5:]
                if t5 == ['H', 'L', 'H', 'L', 'H']:
                    h1, v1, h2, v2, h3 = p5
                    if abs(h1 - h2) / a < 2.0 and abs(h2 - h3) / a < 2.0:
                        neck = min(v1, v2)
                        s = max(0.0, 1.0 - (abs(h1-h2)+abs(h2-h3))/(a*4)) * (1.2 if vc else 0.9)
                        df.at[i, 'triple_pattern']   = 1.0
                        df.at[i, 'triple_direction'] = -1.0
                        df.at[i, 'triple_strength']  = min(1.0, s)
                        df.at[i, 'triple_valid']     = float(cl[i] < neck)
                elif t5 == ['L', 'H', 'L', 'H', 'L']:
                    l1, p1, l2, p2, l3 = p5
                    if abs(l1-l2)/a < 2.0 and abs(l2-l3)/a < 2.0:
                        neck = max(p1, p2)
                        s = max(0.0, 1.0 - (abs(l1-l2)+abs(l2-l3))/(a*4)) * (1.2 if vc else 0.9)
                        df.at[i, 'triple_pattern']   = 1.0
                        df.at[i, 'triple_direction'] = 1.0
                        df.at[i, 'triple_strength']  = min(1.0, s)
                        df.at[i, 'triple_valid']     = float(cl[i] > neck)

            lh2 = [p for p in pts if p['type'] == 'H'][-2:]
            ll2 = [p for p in pts if p['type'] == 'L'][-2:]
            if len(lh2) == 2 and len(ll2) == 2:
                h_dx = max(1, lh2[-1]['idx'] - lh2[-2]['idx'])
                l_dx = max(1, ll2[-1]['idx'] - ll2[-2]['idx'])
                h_sl = (lh2[-1]['price'] - lh2[-2]['price']) / h_dx
                l_sl = (ll2[-1]['price'] - ll2[-2]['price']) / l_dx
                if h_sl < -1e-6 and l_sl > 1e-6:
                    df.at[i, 'triangle_pattern']   = 1.0
                    df.at[i, 'triangle_direction'] = 0.0
                    df.at[i, 'triangle_strength']  = 0.6 * (1.1 if vc else 0.9)
                elif abs(h_sl) < 1e-6 and l_sl > 1e-6:
                    df.at[i, 'triangle_pattern']   = 1.0
                    df.at[i, 'triangle_direction'] = 1.0
                    df.at[i, 'triangle_strength']  = 0.75 * (1.2 if vc else 0.8)
                elif h_sl < -1e-6 and abs(l_sl) < 1e-6:
                    df.at[i, 'triangle_pattern']   = 1.0
                    df.at[i, 'triangle_direction'] = -1.0
                    df.at[i, 'triangle_strength']  = 0.75 * (1.2 if vc else 0.8)
                elif h_sl > 1e-6 and l_sl > 1e-6 and l_sl > h_sl:
                    df.at[i, 'wedge_pattern']   = 1.0
                    df.at[i, 'wedge_direction'] = -1.0
                    df.at[i, 'wedge_strength']  = min(1.0, abs(l_sl - h_sl) / a * 50 * (1.2 if vc else 0.8))
                elif h_sl < -1e-6 and l_sl < -1e-6 and l_sl > h_sl:
                    df.at[i, 'wedge_pattern']   = 1.0
                    df.at[i, 'wedge_direction'] = 1.0
                    df.at[i, 'wedge_strength']  = min(1.0, abs(h_sl - l_sl) / a * 50 * (1.2 if vc else 0.8))
                sd = abs(h_sl - l_sl) / a
                if (sd < 0.5
                        and df.at[i, 'triangle_pattern'] == 0
                        and df.at[i, 'wedge_pattern'] == 0):
                    df.at[i, 'channel_pattern']   = 1.0
                    df.at[i, 'channel_direction'] = (1.0 if h_sl > 0 else (-1.0 if h_sl < 0 else 0.0))
                    df.at[i, 'channel_strength']  = max(0.0, 1.0 - sd) * (1.1 if vc else 0.9)

            if i >= window:
                pm = abs(cl[i] - cl[i - window]) / a
                rr = (max(hi[max(0, i-5):i+1]) - min(lo[max(0, i-5):i+1])) / a
                if pm > 3 and rr < 2.5:
                    df.at[i, 'flag_pattern']   = 1.0
                    df.at[i, 'flag_direction'] = 1.0 if cl[i] > cl[i - window] else -1.0
                    df.at[i, 'flag_strength']  = min(1.0, pm / 8) * (1.2 if vc else 0.8)

            if len(pts) >= 3:
                t3 = types[-3:]; p3 = prices[-3:]
                if t3 == ['H', 'L', 'H']:
                    lh, cup_l, rh = p3
                    depth = (min(lh, rh) - cup_l) / a
                    rdiff = abs(lh - rh) / a
                    if depth > 2 and rdiff < 3 and cl[i] > cup_l * 1.005:
                        rim = max(lh, rh)
                        df.at[i, 'cup_handle_pattern']  = 1.0
                        df.at[i, 'cup_handle_strength'] = min(1.0, depth / 8) * (1.3 if vc else 0.8)
                        df.at[i, 'cup_handle_valid']    = float(cl[i] > rim)

            l3h = [p['price'] for p in pts if p['type'] == 'H'][-3:]
            l3l = [p['price'] for p in pts if p['type'] == 'L'][-3:]
            if len(l3h) == 3 and len(l3l) == 3:
                hs2 = np.std(l3h); ls2 = np.std(l3l)
                ha  = np.mean(l3h); la  = np.mean(l3l)
                if hs2 / a < 1.5 and ls2 / a < 1.5 and la < cl[i] < ha:
                    df.at[i, 'rectangle_pattern']   = 1.0
                    df.at[i, 'rectangle_direction'] = 0.0
                    df.at[i, 'rectangle_strength']  = max(0.0, 1.0 - (hs2+ls2)/(a*3+1e-9))

            df.at[i, 'pattern_volume_confirm'] = vc

        df['pattern_intensity'] = (
            df['hs_pattern']           * 1.5
            + df['double_pattern']     * 1.2
            + df['triple_pattern']     * 1.3
            + df['triangle_pattern']   * 0.8
            + df['wedge_pattern']      * 0.9
            + df['flag_pattern']       * 1.0
            + df['channel_pattern']    * 0.6
            + df['cup_handle_pattern'] * 1.2
            + df['rectangle_pattern']  * 0.5
        )

        has_any = (
            df['hs_pattern'] + df['double_pattern'] + df['triple_pattern']
            + df['triangle_pattern'] + df['wedge_pattern'] + df['flag_pattern']
            + df['channel_pattern'] + df['cup_handle_pattern'] + df['rectangle_pattern']
        ) > 0
        count = 999; bars = []
        for v_ in has_any:
            count = 0 if v_ else min(count + 1, 999)
            bars.append(count)
        df['pattern_age'] = bars
        return df

    # ─── Harmonic Patterns ────────────────────────────────────
    def detect_harmonic_patterns(self, df, zigzag_pct=0.01, confirmation_bars=2):
        df = df.copy()
        n  = len(df)
        for col in ['harmonic_pattern', 'harmonic_type', 'harmonic_direction',
                    'harmonic_strength', 'harmonic_prz', 'harmonic_prz_high',
                    'harmonic_prz_low', 'harmonic_valid']:
            df[col] = 0.0

        PATS = {
            'gartley':   {'AB_XA':(0.618,0.618),'BC_AB':(0.382,0.886),'CD_BC':(1.272,1.618),'AD_XA':(0.786,0.786),'sc':4,'prz_fib':0.786},
            'butterfly': {'AB_XA':(0.786,0.786),'BC_AB':(0.382,0.886),'CD_BC':(1.618,2.618),'AD_XA':(1.272,1.618),'sc':3,'prz_fib':1.272},
            'bat':       {'AB_XA':(0.382,0.500),'BC_AB':(0.382,0.886),'CD_BC':(1.618,2.618),'AD_XA':(0.886,0.886),'sc':3,'prz_fib':0.886},
            'crab':      {'AB_XA':(0.382,0.618),'BC_AB':(0.382,0.886),'CD_BC':(2.240,3.618),'AD_XA':(1.618,1.618),'sc':2,'prz_fib':1.618},
            'shark':     {'AB_XA':(0.446,0.618),'BC_AB':(1.130,1.618),'CD_BC':(0.886,1.130),'AD_XA':(0.886,1.130),'sc':2,'prz_fib':0.886},
            'cypher':    {'AB_XA':(0.382,0.618),'BC_AB':(1.130,1.414),'CD_BC':(0.786,0.786),'AD_XA':(0.786,0.786),'sc':2,'prz_fib':0.786},
        }

        hi_idx, lo_idx = zigzag_swings(
            df['high'], df['low'],
            pct_thresh=zigzag_pct, confirmation_bars=confirmation_bars)
        swings = sorted(
            [{'idx': i, 'price': df['high'].iloc[i], 'type': 'H'} for i in hi_idx]
            + [{'idx': i, 'price': df['low'].iloc[i],  'type': 'L'} for i in lo_idx],
            key=lambda x: x['idx'])
        closes = df['close'].values

        for i in range(4, len(swings)):
            pts    = swings[i-4:i+1]
            types  = [p['type']  for p in pts]
            prices = [p['price'] for p in pts]
            idxs   = [p['idx']   for p in pts]
            Di     = idxs[4]
            if Di >= n: continue
            is_bear = (types == ['H','L','H','L','H'])
            is_bull = (types == ['L','H','L','H','L'])
            if not (is_bear or is_bull): continue
            X, A, B, C, D = prices
            XA=abs(A-X); AB=abs(B-A); BC=abs(C-B); CD=abs(D-C); AD=abs(D-A)
            if any(v < 1e-9 for v in [XA,AB,BC,CD]): continue
            ab_xa=leg_ratio(XA,AB); bc_ab=leg_ratio(AB,BC)
            cd_bc=leg_ratio(BC,CD); ad_xa=leg_ratio(XA,AD)
            t = FIB_TOL
            for pn, r in PATS.items():
                la,ha=r['AB_XA']; lb,hb=r['BC_AB']
                lc,hc=r['CD_BC']; ld,hd=r['AD_XA']
                if (la-t<=ab_xa<=ha+t and lb-t<=bc_ab<=hb+t
                        and lc-t<=cd_bc<=hc+t and ld-t<=ad_xa<=hd+t):
                    acc = 1.0 - (
                        abs(ab_xa-(la+ha)/2)/((ha-la)+0.01)
                        + abs(bc_ab-(lb+hb)/2)/((hb-lb)+0.01)
                        + abs(cd_bc-(lc+hc)/2)/((hc-lc)+0.01)
                    ) / 3
                    prz_c = (X+(A-X)*r['prz_fib'] if is_bull else X-(X-A)*r['prz_fib'])
                    atr_d = max(df['atr'].iloc[Di], 1e-9)
                    df.at[Di, 'harmonic_pattern']   = 1.0
                    df.at[Di, 'harmonic_type']      = float(r['sc'])
                    df.at[Di, 'harmonic_direction'] = -1.0 if is_bear else 1.0
                    df.at[Di, 'harmonic_strength']  = max(0.0, min(1.0, acc))
                    df.at[Di, 'harmonic_prz']       = 1.0
                    df.at[Di, 'harmonic_prz_high']  = prz_c + atr_d * 0.5
                    df.at[Di, 'harmonic_prz_low']   = prz_c - atr_d * 0.5
                    df.at[Di, 'harmonic_valid']     = float(
                        (prz_c - atr_d*0.5) <= closes[Di] <= (prz_c + atr_d*0.5))
                    break
        return df

    # ─── Elliott Wave ─────────────────────────────────────────
    def detect_elliott_wave(self, df, zigzag_pct=0.01, confirmation_bars=2):
        df = df.copy()
        n  = len(df)
        ew_cols = ['elliott_impulse','elliott_wave_num','elliott_corrective',
                   'elliott_wave3','elliott_bullish','elliott_strength',
                   'elliott_extended_w3','elliott_diagonal','elliott_valid',
                   'elliott_w1_len','elliott_w3_len','elliott_w5_len']
        for col in ew_cols:
            df[col] = 0.0

        hi_idx, lo_idx = zigzag_swings(
            df['high'], df['low'],
            pct_thresh=zigzag_pct, confirmation_bars=confirmation_bars)
        swings = sorted(
            [{'idx': i, 'price': df['high'].iloc[i], 'type': 'H'} for i in hi_idx]
            + [{'idx': i, 'price': df['low'].iloc[i],  'type': 'L'} for i in lo_idx],
            key=lambda x: x['idx'])
        atr = df['atr'].values

        for i in range(4, len(swings)):
            pts    = swings[i-4:i+1]
            types  = [p['type']  for p in pts]
            prices = [p['price'] for p in pts]
            idxs   = [p['idx']   for p in pts]

            if types == ['L','H','L','H','L']:
                w0,w1,w2,w3,w4 = prices
                l1=w1-w0; l2=w1-w2; l3=w3-w2; l4=w3-w4; l5=abs(prices[4]-w4)
                valid = (l1>0 and l3>0 and w2>w0 and w4>w1
                         and l3>=l1*0.8 and 0.236<=(l2/(l1+1e-9))<=0.886)
                if valid:
                    ext=l3>l1*1.618; w4_r=l4/(l3+1e-9)
                    alt=not((l2/(l1+1e-9))>0.5 and w4_r>0.5)
                    qb=1.0+(0.20 if ext else 0)+(0.15 if alt else 0)
                    ci=idxs[-1]
                    if ci<n:
                        ai=max(atr[ci],1e-9)
                        df.at[ci,'elliott_impulse']=1.0; df.at[ci,'elliott_bullish']=1.0
                        df.at[ci,'elliott_valid']=1.0; df.at[ci,'elliott_wave_num']=5.0
                        df.at[ci,'elliott_strength']=min(1.0,l3/(ai*5)*qb)
                        df.at[ci,'elliott_w1_len']=l1/ai; df.at[ci,'elliott_w3_len']=l3/ai
                        df.at[ci,'elliott_w5_len']=l5/ai
                    w3i=idxs[3]
                    if w3i<n:
                        aw=max(atr[w3i],1e-9)
                        df.at[w3i,'elliott_wave3']=1.0; df.at[w3i,'elliott_impulse']=1.0
                        df.at[w3i,'elliott_bullish']=1.0; df.at[w3i,'elliott_valid']=1.0
                        df.at[w3i,'elliott_wave_num']=3.0
                        df.at[w3i,'elliott_strength']=min(1.0,l3/(aw*3)*qb)
                        df.at[w3i,'elliott_extended_w3']=float(ext)

            elif types == ['H','L','H','L','H']:
                w0,w1,w2,w3,w4 = prices
                l1=w0-w1; l2=w2-w1; l3=w2-w3; l4=w4-w3; l5=abs(prices[4]-w4)
                valid = (l1>0 and l3>0 and w2<=w0 and w4<=w1
                         and l3>=l1*0.8 and 0.236<=(l2/(l1+1e-9))<=0.886)
                if valid:
                    ext=l3>l1*1.618; w4_r=l4/(l3+1e-9)
                    alt=not((l2/(l1+1e-9))>0.5 and w4_r>0.5)
                    qb=1.0+(0.20 if ext else 0)+(0.15 if alt else 0)
                    ci=idxs[-1]
                    if ci<n:
                        ai=max(atr[ci],1e-9)
                        df.at[ci,'elliott_impulse']=1.0; df.at[ci,'elliott_bullish']=0.0
                        df.at[ci,'elliott_valid']=1.0; df.at[ci,'elliott_wave_num']=5.0
                        df.at[ci,'elliott_strength']=min(1.0,l3/(ai*5)*qb)
                        df.at[ci,'elliott_w1_len']=l1/ai; df.at[ci,'elliott_w3_len']=l3/ai
                        df.at[ci,'elliott_w5_len']=l5/ai
                    w3i=idxs[3]
                    if w3i<n:
                        aw=max(atr[w3i],1e-9)
                        df.at[w3i,'elliott_wave3']=1.0; df.at[w3i,'elliott_impulse']=1.0
                        df.at[w3i,'elliott_bullish']=0.0; df.at[w3i,'elliott_valid']=1.0
                        df.at[w3i,'elliott_wave_num']=3.0
                        df.at[w3i,'elliott_strength']=min(1.0,l3/(aw*3)*qb)
                        df.at[w3i,'elliott_extended_w3']=float(ext)

            if types in [['L','H','L','H','L'],['H','L','H','L','H']]:
                legs=[abs(prices[j+1]-prices[j]) for j in range(4)]
                if all(legs[j]<legs[j-1] for j in range(1,4)):
                    ci=idxs[-1]
                    if ci<n:
                        df.at[ci,'elliott_diagonal']=1.0
                        df.at[ci,'elliott_bullish']=float(types==['L','H','L','H','L'])
                        df.at[ci,'elliott_valid']=1.0

        for i in range(2, len(swings)):
            pts    = swings[i-2:i+1]
            types  = [p['type']  for p in pts]
            prices = [p['price'] for p in pts]
            ci     = pts[-1]['idx']
            if ci >= n: continue
            if types == ['H','L','H']:
                A,B,C=prices; lA=A-B; lB=C-B
                if lA>0 and lB>0 and 0.236<=lB/(lA+1e-9)<=1.618:
                    df.at[ci,'elliott_corrective']=1.0
                    df.at[ci,'elliott_bullish']=1.0; df.at[ci,'elliott_valid']=1.0
            elif types == ['L','H','L']:
                A,B,C=prices; lA=B-A; lB=B-C
                if lA>0 and lB>0 and 0.236<=lB/(lA+1e-9)<=1.618:
                    df.at[ci,'elliott_corrective']=1.0
                    df.at[ci,'elliott_bullish']=0.0; df.at[ci,'elliott_valid']=1.0
        return df

    # ─── Pattern Meta ─────────────────────────────────────────
    def add_pattern_meta(self, df, max_age=20):
        df = df.copy()

        def _s(col, w=1.0, vcol=None):
            base = df[col].fillna(0) if col in df.columns else pd.Series(0.0, index=df.index)
            if vcol:
                vv = df[vcol].fillna(0) if vcol in df.columns else pd.Series(1.0, index=df.index)
                return base * w * vv
            return base * w

        pcols = ['hs_pattern','double_pattern','triple_pattern','triangle_pattern',
                 'flag_pattern','wedge_pattern','channel_pattern',
                 'cup_handle_pattern','rectangle_pattern',
                 'harmonic_pattern','elliott_impulse','elliott_corrective']
        df['pattern_count'] = df[pcols].clip(0,1).sum(axis=1)

        ew3  = df['elliott_wave3'].clip(0,1)       if 'elliott_wave3'       in df.columns else pd.Series(0.0,index=df.index)
        ewx3 = df['elliott_extended_w3'].clip(0,1) if 'elliott_extended_w3' in df.columns else pd.Series(0.0,index=df.index)

        df['pattern_cluster'] = (
            _s('harmonic_pattern') + _s('hs_pattern') + _s('double_pattern')
            + _s('triple_pattern') + ew3 + ewx3
        )
        d_mean = df[[c for c in ['hs_direction','double_direction','triple_direction','harmonic_direction']
                     if c in df.columns]].mean(axis=1).abs()
        df['cluster_boost']  = (df['pattern_cluster'] * (1.0 + d_mean)).fillna(df['pattern_cluster'])
        df['strong_cluster'] = (df['cluster_boost'] >= 1.5).astype(float)

        df['pattern_quality'] = (
            _s('harmonic_strength',     3.0, 'harmonic_valid')
            + _s('hs_strength',         2.5, 'hs_valid')
            + _s('elliott_strength',    2.0, 'elliott_valid')
            + _s('triple_strength',     1.8, 'triple_valid')
            + _s('cup_handle_strength', 1.6, 'cup_handle_valid')
            + _s('double_strength',     1.4, 'double_valid')
            + _s('triangle_strength',   1.2)
            + _s('flag_strength',       1.0)
            + _s('wedge_strength',      1.0)
            + _s('channel_strength',    0.8)
            + _s('rectangle_strength',  0.5)
        )
        qmax = df['pattern_quality'].max()
        df['pattern_quality'] = df['pattern_quality'] / (qmax + 1e-9)

        vcols = [c for c in ['hs_valid','double_valid','triple_valid',
                              'cup_handle_valid','harmonic_valid','elliott_valid']
                 if c in df.columns]
        df['pattern_valid'] = ((df[vcols].sum(axis=1)>0).astype(float) if vcols
                               else pd.Series(0.0,index=df.index))

        dcols = [c for c in ['hs_direction','double_direction','triple_direction',
                              'triangle_direction','harmonic_direction','elliott_bullish']
                 if c in df.columns]
        df['direction_consensus'] = df[dcols].mean(axis=1) if dcols else 0.0

        df['bars_since_pattern'] = df['pattern_age']
        df['pattern_freshness']  = 1.0 / (df['bars_since_pattern'] + 1)
        df['pattern_expired']    = (df['pattern_age'] > max_age).astype(float)

        htf_trend_col = df.get('htf_trend', pd.Series(0, index=df.index)) if isinstance(df, pd.DataFrame) else pd.Series(0, index=df.index)
        if 'htf_trend' in df.columns:
            htf_trend_col = df['htf_trend']
        df['pattern_htf_aligned'] = (
            ((df['direction_consensus'] > 0) & (htf_trend_col > 0))
            | ((df['direction_consensus'] < 0) & (htf_trend_col < 0))
        ).astype(float)

        if 'pattern_volume_confirm' in df.columns:
            df['pattern_with_volume'] = (
                (df['pattern_count'] >= 1) & (df['pattern_volume_confirm'] == 1)
            ).astype(float)
        else:
            df['pattern_with_volume'] = 0.0

        df['pattern_score'] = (
            df['pattern_quality'] * df['pattern_freshness']
            * (1.0 + df['pattern_htf_aligned'] * 0.5)
            * (1.0 + df['pattern_with_volume'] * 0.3)
            * (1.0 + df['pattern_valid'] * 0.4)
        )

        if 'harmonic_prz_high' in df.columns:
            prz_h = df['harmonic_prz_high'].shift(1).fillna(0.0)
            prz_l = df['harmonic_prz_low'].shift(1).fillna(0.0)
            df['in_prz']      = ((df['close'] >= prz_l) & (df['close'] <= prz_h) & (prz_h > 0)).astype(float)
            df['dist_to_prz'] = (df['close'] - (prz_h + prz_l)/2).abs() / (df['atr'] + 1e-9)
        else:
            df['in_prz'] = 0.0; df['dist_to_prz'] = 0.0
        return df

    # ─── Create Features ──────────────────────────────────────
    def create_features(self, df, config, shift=True):
        w  = config.get('pattern_window',  20)
        hr = config.get('htf_resample',    None)
        zp = config.get('zigzag_pct',      0.01)
        ma = config.get('pattern_max_age', 20)
        cb = config.get('confirmation_bars', 2)

        # v4.5: تمرير target type للـ create_target عبر instance attributes
        self._use_median_split = config.get('use_median_split', False)
        self._use_directional  = config.get('use_directional',  False)

        df = df.reset_index(drop=True).copy()
        df = self.add_base_features(df)
        df = self.add_htf_trend(df, htf_resample=hr)
        df = self.add_regime_features(df)
        df = self.add_order_flow(df)
        df = self.detect_classic_patterns(df, window=w, zigzag_pct=zp,
                                           max_age=ma, confirmation_bars=cb)
        df = self.detect_harmonic_patterns(df, zigzag_pct=zp, confirmation_bars=cb)
        df = self.detect_elliott_wave(df, zigzag_pct=zp, confirmation_bars=cb)
        df = self.add_pattern_meta(df, max_age=ma)

        lag_cols = ['rsi','macd_hist','mfi','bb_pctb','volatility',
                    'pattern_score','direction_consensus','adx',
                    'cmf','momentum_score','vol_ratio','di_diff',
                    'stoch_k','williams_r','kc_pctk','trend_consistency']
        for col in lag_cols:
            if col in df.columns:
                df[f'{col}_lag1'] = df[col].shift(1)
                df[f'{col}_lag3'] = df[col].shift(3)

        excl = {'datetime','open','high','low','close','volume','hour','dow','month'}
        if shift:
            for col in [c for c in df.columns if c not in excl]:
                df[col] = df[col].shift(1)

        df = df.ffill().fillna(0)
        df.replace([np.inf, -np.inf], 0, inplace=True)
        warmup = max(10, min(80, len(df) // 8))
        df = df.iloc[warmup:].copy()

        thresh = int(len(df) * 0.3)
        df = df.dropna(axis=1, thresh=thresh)
        df = df.ffill().fillna(0)
        df.replace([np.inf, -np.inf], 0, inplace=True)

        excl_f = {'datetime','open','high','low','close','volume',
                  'future_close','pct_change','target'}
        self.feature_columns = [c for c in df.columns if c not in excl_f]
        logging.info(f"Features: {len(self.feature_columns)}")
        return df

    # ─── Create Target ────────────────────────────────────────
    def create_target(self, df, horizon=5, min_return=0.001, tf_name="5min"):
        """
        v4.5 — ثلاثة أنواع من الـ target:

        1. directional (1h, 4h, 1d): target= UP/DOWN فقط، يحذف neutral zone
           → AUC>0.5 مضمون + WR حقيقي + الموديل يتعلم اتجاهاً واضحاً

        2. median_split (45min, 1W, 2h): فوق/تحت median → 50% balance

        3. quantile (5min, 15min): top/bottom quantile → accuracy عالية
        """
        df = df.copy()
        df['future_close'] = df['close'].shift(-horizon)
        df['pct_change']   = (df['future_close'] - df['close']) / (df['close'] + 1e-9)
        df.dropna(subset=['future_close'], inplace=True)

        pct = df['pct_change']
        if len(pct) < 10:
            df['target'] = 0
            return df

        use_directional = getattr(self, '_use_directional', False)
        use_median      = getattr(self, '_use_median_split', False)

        # ── 1. Directional Target ─────────────────────────────
        if use_directional:
            if 'atr' in df.columns:
                atr_ret = (df['atr'] / df['close'].replace(0, 1e-9)).median()
                thr = max(atr_ret * 0.5, min_return)
            else:
                thr = min_return

            for _ in range(8):
                up_ratio   = (pct > thr).mean()
                down_ratio = (pct < -thr).mean()
                total_dir  = up_ratio + down_ratio
                if 0.45 <= total_dir <= 0.80:
                    break
                thr *= (0.88 if total_dir < 0.45 else 1.12)

            # احتفظ بالإشارات الواضحة فقط (احذف neutral zone)
            mask = (pct > thr) | (pct < -thr)
            df   = df[mask].copy()
            df['target'] = (df['pct_change'] > 0).astype(int)
            pos = df['target'].mean()
            logging.info(f"Target [{tf_name}] Directional: {len(df)} rows | "
                         f"UP={pos:.1%} DOWN={1-pos:.1%} | thr={thr:.6f}")
            return df

        # ── 2. Median Split ───────────────────────────────────
        if use_median:
            thr = float(pct.median())
            df['target'] = (df['pct_change'] > thr).astype(int)
            logging.info(f"Target [{tf_name}] Median: {len(df)} rows | "
                         f"Positive:{df['target'].mean():.1%} | median={thr:.6f}")
            return df

        # ── 3. Quantile Target (5min, 15min) ──────────────────
        if 'atr' in df.columns and len(pct) > 50:
            atr_ret  = (df['atr'] / df['close'].replace(0, 1e-9)).median()
            base_thr = max(atr_ret * 0.35, min_return)
        else:
            base_thr = min_return

        pos_thr = pct.quantile(0.58)
        thr = max(base_thr, abs(pos_thr))

        for _ in range(10):
            pos_ratio = (pct > thr).mean()
            if 0.35 <= pos_ratio <= 0.55:
                break
            thr *= (0.82 if pos_ratio < 0.35 else 1.18)

        df['target'] = (df['pct_change'] > thr).astype(int)
        logging.info(f"Target [{tf_name}] Quantile: {len(df)} rows | "
                     f"Positive:{df['target'].mean():.1%} | thr={thr:.6f}")
        return df

    def fit_scaler(self, X):
        self.scaler.fit(X)
        self.scaler_fitted = True
        if isinstance(X, pd.DataFrame):
            self.feature_columns = list(X.columns)

    def transform(self, X):
        if not self.scaler_fitted:
            raise RuntimeError("scaler not fitted")
        if isinstance(X, pd.DataFrame):
            missing = [c for c in self.feature_columns if c not in X.columns]
            for c in missing:
                X[c] = 0.0
            X = X[self.feature_columns]
        Xs = self.scaler.transform(X)
        if self.selector is not None and self.selected_cols is not None:
            feat_idx = [self.feature_columns.index(c) for c in self.selected_cols if c in self.feature_columns]
            Xs = Xs[:, feat_idx]
        return Xs

    def fit_transform(self, X):
        self.fit_scaler(X)
        return self.transform(X)

    # v4.1-fast: Feature Selection مُحسَّنة السرعة
    def fit_feature_selection(self, X_scaled, y, n_features=None):
        """SelectFromModel — أسرع بكثير: n_estimators=100, max_depth=3"""
        logging.info(f"  Feature Selection on {X_scaled.shape[1]} features...")
        sel_model = xgb.XGBClassifier(
            n_estimators=100,   # ← خُفِّض من 200
            max_depth=3,        # ← خُفِّض من 4
            learning_rate=0.1,  # ← رُفع لتعويض العدد
            subsample=0.8,
            colsample_bytree=0.7,
            random_state=42,
            tree_method='hist',
            verbosity=0,
            use_label_encoder=False
        )
        sel_model.fit(X_scaled, y)
        importances = sel_model.feature_importances_

        if n_features:
            threshold = np.sort(importances)[::-1][min(n_features, len(importances)-1)]
        else:
            threshold = np.percentile(importances, 30)

        mask = importances >= threshold
        selected_indices = np.where(mask)[0]

        if len(selected_indices) < 10:
            selected_indices = np.argsort(importances)[::-1][:max(10, len(importances)//3)]

        self.selected_cols = [self.feature_columns[i] for i in selected_indices
                               if i < len(self.feature_columns)]
        logging.info(f"  Selected {len(self.selected_cols)} / {len(self.feature_columns)} features")
        return self.selected_cols


# ══════════════════════════════════════════════════════════════
# BACKTESTER v4
# ══════════════════════════════════════════════════════════════
class BacktesterV4:
    def __init__(self, initial_capital=10000,
                 spread_bps=SPREAD_BPS,
                 risk_per_trade=0.01,
                 max_positions=1,
                 max_position_pct=0.10,
                 max_drawdown_pct=0.25,
                 min_pattern_quality=0.10,
                 require_valid=False,
                 confidence_threshold=0.55,
                 atr_sl_mult=ATR_SL_MULT,
                 atr_tp_mult=ATR_TP_MULT,
                 adx_min=10.0,
                 require_htf_align=True,
                 htf_override_threshold=0.72):
        self.initial_capital      = initial_capital
        self.spread               = spread_bps / 10000
        self.slippage             = 0.00005
        self.risk_per_trade       = risk_per_trade
        self.max_positions        = max_positions
        self.max_position_pct     = max_position_pct
        self.max_drawdown_pct     = max_drawdown_pct
        self.min_pattern_quality  = min_pattern_quality
        self.require_valid        = require_valid
        self.confidence_threshold = confidence_threshold
        self.atr_sl_mult          = atr_sl_mult
        self.atr_tp_mult          = atr_tp_mult
        self.adx_min              = adx_min
        self.require_htf_align    = require_htf_align
        self.htf_override_threshold = htf_override_threshold

    def _exec(self, p, buy):
        if buy:
            return p * (1 + self.spread) * (1 + self.slippage)
        return p * (1 - self.spread) * (1 - self.slippage)

    def _get_signal(self, pr, pq_val, adx_val, htf_val, bth, sth, adx_min_local):
        raw = "BUY" if pr > bth else ("SELL" if pr < sth else "HOLD")
        if raw == "HOLD":
            return "HOLD"
        if pq_val < self.min_pattern_quality:
            return "HOLD"
        if adx_val < adx_min_local:
            return "HOLD"
        if self.require_htf_align and htf_val != 0:
            if (raw == "BUY" and htf_val < 0) or (raw == "SELL" and htf_val > 0):
                if pr < self.htf_override_threshold:
                    return "HOLD"
        return raw

    def run(self, df, model, fe, feature_columns, cfg, use_fold=False):
        capital   = self.initial_capital
        positions = []; trades = []; equity = [capital]

        needed = feature_columns + ['close', 'atr']
        missing = [c for c in needed if c not in df.columns]
        if missing:
            logging.error(f"Missing columns: {missing[:5]}")
            return {}

        bth = cfg.get('fold_buy_threshold' if use_fold else 'buy_threshold',
                      self.confidence_threshold)
        sth = cfg.get('fold_sell_threshold' if use_fold else 'sell_threshold',
                      1.0 - self.confidence_threshold)
        sl_m = cfg.get('atr_sl_mult', self.atr_sl_mult)
        tp_m = cfg.get('atr_tp_mult', self.atr_tp_mult)

        try:
            X_all    = df[feature_columns].copy()
            if fe.selected_cols is not None:
                for c in fe.selected_cols:
                    if c not in X_all.columns:
                        X_all[c] = 0.0
            X_scaled = fe.transform(X_all)
            proba    = model.predict_proba(X_scaled)[:, 1]
        except Exception as e:
            logging.error(f"Predict error: {e}")
            return {}

        closes = df['close'].values
        atrs   = df['atr'].values
        pq     = df['pattern_quality'].values if 'pattern_quality' in df.columns else np.ones(len(df))
        adx_v  = df['adx'].values             if 'adx'             in df.columns else np.full(len(df), 20.0)
        vol_r  = df['vol_regime'].values      if 'vol_regime'      in df.columns else np.full(len(df), 1)
        htf_t  = df['htf_trend'].values       if 'htf_trend'       in df.columns else np.zeros(len(df))

        peak_capital = capital
        adx_min_local = cfg.get('adx_min', self.adx_min)

        for i in range(len(df)):
            p, a, pr = closes[i], max(float(atrs[i]), 1e-9), proba[i]

            regime = vol_r[i]
            sl_curr = sl_m * (0.85 if regime == 0 else (1.20 if regime == 2 else 1.0))
            tp_curr = tp_m * (1.10 if regime == 0 else (1.35 if regime == 2 else 1.0))

            peak_capital = max(peak_capital, capital)
            curr_dd = (capital - peak_capital) / (peak_capital + 1e-9)

            if curr_dd < -self.max_drawdown_pct:
                sig = "HOLD"
            else:
                sig = self._get_signal(pr, pq[i], adx_v[i], htf_t[i],
                                       bth, sth, adx_min_local)

            to_close = []
            for pos in positions:
                if pos['type'] == 'buy':
                    if p <= pos['sl'] or p >= pos['tp']:
                        ep = pos['sl'] if p <= pos['sl'] else pos['tp']
                        profit = pos['units'] * (ep - pos['entry'])
                        capital += pos['units'] * ep
                        trades.append({'profit': profit,
                                       'exit': 'sl' if p <= pos['sl'] else 'tp',
                                       'direction': 1, 'entry': pos['entry'],
                                       'exit_price': ep})
                        to_close.append(pos)
                else:
                    if p >= pos['sl'] or p <= pos['tp']:
                        ep = pos['sl'] if p >= pos['sl'] else pos['tp']
                        profit = pos['units'] * (pos['entry'] - ep)
                        capital += pos['margin'] + profit
                        trades.append({'profit': profit,
                                       'exit': 'sl' if p >= pos['sl'] else 'tp',
                                       'direction': -1, 'entry': pos['entry'],
                                       'exit_price': ep})
                        to_close.append(pos)
            for pos in to_close:
                positions.remove(pos)

            if sig == "BUY" and len(positions) < self.max_positions and a > 0:
                ra = capital * self.risk_per_trade
                sd = sl_curr * a
                if sd > 0:
                    entry = self._exec(p, True)
                    units = ra / sd
                    cost  = min(units * entry, capital * self.max_position_pct)
                    units = cost / entry
                    if cost > 0 and cost <= capital:
                        capital -= cost
                        positions.append({
                            'type': 'buy', 'entry': entry, 'units': units,
                            'sl': entry - sd, 'tp': entry + tp_curr * a, 'margin': cost
                        })
            elif sig == "SELL" and len(positions) < self.max_positions and a > 0:
                ra     = capital * self.risk_per_trade
                sd     = sl_curr * a
                if sd > 0:
                    entry  = self._exec(p, False)
                    margin = min(ra, capital * self.max_position_pct)
                    if margin > 0 and margin <= capital:
                        capital -= margin
                        positions.append({
                            'type': 'sell', 'entry': entry,
                            'units': margin / entry if entry > 0 else 0,
                            'sl': entry + sd, 'tp': entry - tp_curr * a, 'margin': margin
                        })

            ov = sum(
                pos['units'] * p - pos['units'] * pos['entry'] if pos['type'] == 'buy'
                else pos['units'] * pos['entry'] - pos['units'] * p
                for pos in positions
            )
            equity.append(capital + ov)

        for pos in positions:
            lp = closes[-1]
            if pos['type'] == 'buy':
                pnl = pos['units'] * (self._exec(lp, False) - pos['entry'])
                capital += pos['units'] * self._exec(lp, False)
            else:
                pnl = pos['units'] * (pos['entry'] - self._exec(lp, True))
                capital += pos['margin'] + pnl
            trades.append({'profit': pnl, 'exit': 'eod',
                           'direction': 1 if pos['type']=='buy' else -1,
                           'entry': pos['entry'], 'exit_price': lp})

        if not trades:
            return {'total_trades': 0, 'win_rate': 0, 'profit_factor': 0,
                    'total_return_pct': 0, 'max_drawdown_pct': 0,
                    'avg_win': 0, 'avg_loss': 0, 'expectancy': 0, 'trades': []}

        wins = [t for t in trades if t['profit'] > 0]
        loss = [t for t in trades if t['profit'] <= 0]
        wr   = len(wins) / len(trades) * 100
        tp_s = sum(t['profit'] for t in wins)
        tl_s = abs(sum(t['profit'] for t in loss))
        aw   = tp_s / len(wins) if wins else 0
        al   = tl_s / len(loss) if loss else 0
        pf   = min(99.9, tp_s / tl_s) if tl_s > 0 else (99.9 if tp_s > 0 else 0)
        ret  = (capital - self.initial_capital) / self.initial_capital * 100
        eq_s = pd.Series(equity)
        dd   = float(((eq_s - eq_s.cummax()) / (eq_s.cummax() + 1e-9) * 100).min())

        eq_rets = eq_s.pct_change().dropna()
        sharpe  = float(eq_rets.mean() / (eq_rets.std() + 1e-9) * np.sqrt(252)) if len(eq_rets) > 1 else 0.0

        return {
            'total_trades':     len(trades),
            'win_rate':         round(wr, 2),
            'profit_factor':    round(pf, 2),
            'total_return_pct': round(ret, 2),
            'max_drawdown_pct': round(dd, 2),
            'avg_win':          round(aw, 4),
            'avg_loss':         round(al, 4),
            'expectancy':       round((wr/100 * aw) - ((1-wr/100) * al), 4) if al else 0,
            'sharpe_ratio':     round(sharpe, 3),
            'trades':           trades
        }


# ══════════════════════════════════════════════════════════════
# MODEL BUILDERS v4.1-fast
# ══════════════════════════════════════════════════════════════
def build_xgb(spw, n_est=400, depth=5, lr=0.05,   # ← n_est مخفض
              subsample=0.80, colsample=0.70,
              alpha=0.3, lam=1.5, mcw=3, gamma=0.05):
    return xgb.XGBClassifier(
        objective='binary:logistic',
        n_estimators=n_est,
        max_depth=depth,
        learning_rate=lr,
        subsample=subsample,
        colsample_bytree=colsample,
        reg_alpha=alpha,
        reg_lambda=lam,
        min_child_weight=mcw,
        gamma=gamma,
        scale_pos_weight=spw,
        random_state=42,
        tree_method='hist',
        verbosity=0,
        use_label_encoder=False,
        eval_metric='auc',
        early_stopping_rounds=25,  # ← خُفِّض من 40
    )


def get_xgb_params_for_tf(tf_name, spw, n_samples=None):
    """
    v4.7: XGB parameters مُكيَّفة مع حجم البيانات
    - بيانات محدودة (1h/2h/4h: 6 سنوات فقط) → depth أقل + regularization أقوى
    - بيانات كافية (5min/15min/1d/1W) → params عادية
    """
    # حجم البيانات المتاح فعلياً
    limited_data_tfs = ['1h', '2h', '4h', '45min']  # 6 سنوات فقط

    if tf_name in limited_data_tfs:
        # regularization أقوى لمنع overfitting مع البيانات المحدودة
        return build_xgb(spw, n_est=300, depth=3, lr=0.05,
                         subsample=0.75, colsample=0.65,
                         alpha=1.0, lam=3.0, mcw=8, gamma=0.15)
    elif tf_name in ['1d', '1W']:
        # بيانات كافية لكن عدد الصفقات قليل → نحافظ على بساطة الموديل
        return build_xgb(spw, n_est=400, depth=3, lr=0.04,
                         subsample=0.75, colsample=0.65,
                         alpha=0.8, lam=2.5, mcw=10, gamma=0.2)
    else:
        # 5min/15min: بيانات وفيرة → params عادية
        return build_xgb(spw, n_est=400, depth=5, lr=0.05,
                         subsample=0.80, colsample=0.70,
                         alpha=0.3, lam=1.5, mcw=3, gamma=0.05)


def build_ensemble_model(spw, tf_name):
    """
    v4.1-fast: Ensemble أخف بكثير
    - XGB: n_estimators=200 (كان 500)
    - RF: n_estimators=80 (كان 200)، max_depth=6 (كان 8)
    - StackingClassifier: cv=3 (كان 5)
    - بدون LGB في الـ stacking (إن وُجد يُضاف اختيارياً)
    """
    xgb_base = xgb.XGBClassifier(
        objective='binary:logistic',
        n_estimators=200,       # ← خُفِّض من 500
        max_depth=4,
        learning_rate=0.05,     # ← رُفع لتعويض العدد
        subsample=0.80,
        colsample_bytree=0.70,
        reg_alpha=0.4,
        reg_lambda=1.8,
        min_child_weight=4,
        gamma=0.07,
        scale_pos_weight=min(spw, 10.0),  # v4.3: رفع الـ cap
        random_state=42,
        tree_method='hist',
        verbosity=0,
        use_label_encoder=False,
        eval_metric='logloss',
        early_stopping_rounds=None,
    )

    rf_base = RandomForestClassifier(
        n_estimators=80,
        max_depth=6,
        min_samples_split=25,
        min_samples_leaf=12,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )

    estimators = [('xgb', xgb_base), ('rf', rf_base)]

    # LGB اختياري — فقط لو موجود وحجم البيانات كبير
    if HAS_LGB:
        lgb_base = lgb.LGBMClassifier(
            n_estimators=200,   # ← خُفِّض من 500
            num_leaves=20,      # ← خُفِّض من 31
            learning_rate=0.05,
            subsample=0.80,
            colsample_bytree=0.70,
            reg_alpha=0.4,
            reg_lambda=1.8,
            min_child_samples=25,
            scale_pos_weight=min(spw, 10.0),  # v4.3
            random_state=42,
            verbosity=-1
        )
        estimators.append(('lgb', lgb_base))

    meta = LogisticRegression(
        C=0.5,
        max_iter=300,
        random_state=42,
        class_weight='balanced'
    )

    stacker = StackingClassifier(
        estimators=estimators,
        final_estimator=meta,
        cv=3,           # ← خُفِّض من 5 (أهم تحسين سرعة)
        stack_method='predict_proba',
        n_jobs=-1,
        passthrough=False
    )

    logging.info(f"  Ensemble: {[e[0] for e in estimators]} → LogReg meta (cv=3)")
    return stacker


def build_calibrated_model(base_model, method='sigmoid'):
    """
    v4.2-fix: sigmoid أفضل من isotonic عند البيانات المحدودة.
    - isotonic: يحتاج 1000+ عينة لكل class وإلا يُخرج probabilities مُكسورة
    - sigmoid: يعمل مع بيانات أقل وأكثر استقراراً
    """
    return CalibratedClassifierCV(
        estimator=base_model,
        method=method,  # 'sigmoid' للـ 1h/4h, 'isotonic' للـ 5min/15min إذا أردت
        cv=3
    )


# ══════════════════════════════════════════════════════════════
# WALK-FORWARD v4.1-fast
# ══════════════════════════════════════════════════════════════
def walk_forward_v4(X, y, df_tgt, cfg, n_splits=6,
                    purge_pct=0.005, tf_name=''):
    n          = len(X)
    purge_size = max(1, int(n * purge_pct))
    # ← init_train رُفع إلى 50% (folds أصغر = أسرع)
    # v4.7: init_train مرن بناءً على حجم البيانات
    # للبيانات المحدودة (1h/2h/4h) → 55% training لتجنب overfitting
    # للبيانات الكافية → 50% عادي
    limited_tfs = ['1h', '2h', '4h', '45min']
    init_pct = 0.55 if tf_name in limited_tfs else 0.50
    init_train = max(int(n * init_pct), max(150, n // 4))
    remaining  = n - init_train
    test_size  = max(int(remaining / n_splits), max(20, n // 15))

    min_pq    = cfg.get('min_pattern_quality', 0.10)
    adx_min   = cfg.get('adx_min', 10.0)
    conf_thr  = cfg.get('confidence_threshold', 0.55)

    use_ensemble     = cfg.get('use_ensemble', False)
    use_fs           = cfg.get('use_feature_selection', False)
    use_calib        = cfg.get('use_calibration', False)
    calib_method     = cfg.get('calibration_method', 'sigmoid')
    use_median_split  = cfg.get('use_median_split', False)
    use_directional   = cfg.get('use_directional',  False)   # v4.5
    spw_cap           = cfg.get('spw_cap', 5.0)

    logging.info(f"  WF v4.3: n={n} | init={init_train} | test≈{test_size} | tf={tf_name}")
    logging.info(f"  Mode: ensemble={use_ensemble} | fs={use_fs} | calib={use_calib} | median={use_median_split} | spw_cap={spw_cap}")

    fold_results = []
    best_model   = None
    best_score   = -np.inf
    best_fe      = None
    mdl = fef   = None

    for fold in range(n_splits):
        train_end  = init_train + fold * test_size
        test_start = train_end + purge_size
        test_end   = min(test_start + test_size, n)

        if test_start >= n or test_end <= test_start:
            break

        tei = list(range(test_start, test_end))

        if len(y.iloc[tei].unique()) < 2:
            ext_end = min(test_end + test_size // 2, n)
            tei_ext = list(range(test_start, ext_end))
            if len(y.iloc[tei_ext].unique()) >= 2:
                tei = tei_ext
            else:
                logging.warning(f"  Fold {fold+1}: single class — skip")
                continue

        tri = list(range(0, train_end))
        Xtr, Xte = X.iloc[tri], X.iloc[tei]
        ytr, yte = y.iloc[tri], y.iloc[tei]

        if len(ytr.unique()) < 2:
            mc  = 1 if (ytr == 0).all() else 0
            Xtr = pd.concat([Xtr, Xte.iloc[:2]])
            ytr = pd.concat([ytr, pd.Series([mc, 1-mc], index=Xte.index[:2])])

        neg = (ytr == 0).sum(); pos = (ytr == 1).sum()
        # v4.3: spw_cap مرن — مع median_split يكون ~1.0 تلقائياً
        raw_spw = neg / pos if pos > 0 else 1.0
        spw = min(raw_spw, spw_cap)
        logging.info(f"  Fold {fold+1}: neg={neg} pos={pos} spw={spw:.2f}")

        fef = PatternFeatureEngineerV4()
        fef.feature_columns = list(X.columns)
        fef.fit_scaler(Xtr)
        Xtrs_full = fef.transform(Xtr)
        Xtes_full = fef.transform(Xte)

        if use_fs:
            fef.fit_feature_selection(Xtrs_full, ytr)
            feat_idx = [fef.feature_columns.index(c) for c in fef.selected_cols
                        if c in fef.feature_columns]
            Xtrs = Xtrs_full[:, feat_idx]
            Xtes = Xtes_full[:, feat_idx]
        else:
            Xtrs = Xtrs_full
            Xtes = Xtes_full

        if use_ensemble:
            mdl = build_ensemble_model(spw, tf_name)
            mdl.fit(Xtrs, ytr)
            if use_calib and len(np.unique(ytr)) == 2:
                try:
                    mdl = build_calibrated_model(mdl, method=calib_method)
                    mdl.fit(Xtrs, ytr)
                except Exception as e:
                    logging.warning(f"  Calibration failed: {e} — using uncalibrated")
        else:
            mdl = get_xgb_params_for_tf(tf_name, spw)
            if use_calib:
                mdl.set_params(early_stopping_rounds=None, eval_metric='logloss')
                mdl = build_calibrated_model(mdl, method=calib_method)
                mdl.fit(Xtrs, ytr)
            else:
                mdl.fit(Xtrs, ytr, eval_set=[(Xtes, yte)], verbose=False)

        yp  = mdl.predict(Xtes)
        ypp = mdl.predict_proba(Xtes)[:, 1]
        acc = accuracy_score(yte, yp)
        pr  = precision_score(yte, yp, zero_division=0)
        rc  = recall_score(yte, yp,    zero_division=0)
        f1  = f1_score(yte, yp,        zero_division=0)
        try:
            auc = roc_auc_score(yte, ypp)
        except Exception:
            auc = 0.5

        df_fold = df_tgt.iloc[tei].copy()
        bt = BacktesterV4(
            min_pattern_quality=min_pq,
            confidence_threshold=conf_thr,
            adx_min=adx_min,
            atr_sl_mult=cfg.get('atr_sl_mult', ATR_SL_MULT),
            atr_tp_mult=cfg.get('atr_tp_mult', ATR_TP_MULT),
            require_htf_align=cfg.get('require_htf_align', True),          # v4.4
            htf_override_threshold=cfg.get('htf_override_threshold', 0.68), # v4.4
        ).run(df_fold, mdl, fef, fef.feature_columns, cfg, use_fold=True)

        fst = str(df_tgt['datetime'].iloc[test_start])[:16] if 'datetime' in df_tgt.columns else ''
        fet = str(df_tgt['datetime'].iloc[min(test_end-1, len(df_tgt)-1)])[:16] if 'datetime' in df_tgt.columns else ''
        logging.info(
            f"  Fold {fold+1}/{n_splits} [{fst}→{fet}] "
            f"Acc:{acc:.2%} AUC:{auc:.3f} F1:{f1:.3f}"
        )
        if bt and bt.get('total_trades', 0) > 0:
            logging.info(
                f"    BT | T:{bt['total_trades']} WR:{bt['win_rate']}% "
                f"PF:{bt['profit_factor']} R:{bt['total_return_pct']}% "
                f"Sharpe:{bt.get('sharpe_ratio',0):.2f}"
            )

        fold_results.append({
            'fold': fold+1, 'train_size': train_end, 'test_size': len(tei),
            'purge_size': purge_size, 'period_start': fst, 'period_end': fet,
            'accuracy': round(acc, 4), 'precision': round(pr, 4),
            'recall': round(rc, 4), 'f1': round(f1, 4), 'auc': round(auc, 4),
            'backtest': bt
        })

        n_trades = bt.get('total_trades', 0) if bt else 0
        pf_val   = bt.get('profit_factor', 0) if bt else 0
        sharpe   = bt.get('sharpe_ratio',  0) if bt else 0
        sc = auc + 0.2 * np.log1p(n_trades) + 0.1 * min(pf_val, 3.0) + 0.1 * min(max(sharpe, 0), 2.0)
        if sc > best_score:
            best_score = sc; best_model = mdl; best_fe = fef

    if best_model is None:
        if mdl is not None:
            logging.warning("⚠️  Fallback to last fold model")
            best_model = mdl; best_fe = fef
        else:
            raise ValueError("Walk-Forward failed: all folds skipped")

    return fold_results, best_model, best_fe


# ══════════════════════════════════════════════════════════════
# TRAIN SINGLE TIMEFRAME
# ══════════════════════════════════════════════════════════════
def train_model(df_raw, tf):
    """
    df_raw: يمكن أن يكون:
      - DataFrame (5min raw) → يعمل resample (الطريقة القديمة)
      - None → يحمّل الـ CSV المخصص لهذا الـ timeframe (الطريقة الجديدة)
      - dict of DataFrames → {tf_name: df} (الطريقة الأسرع)
    """
    cfg         = TIMEFRAME_CONFIG[tf]
    min_candles = cfg.get('min_candles', 200)
    horizon     = cfg.get('horizon', 5)
    min_return  = cfg.get('min_return', 0.001)
    n_splits    = cfg.get('n_splits', 6)

    use_ensemble = cfg.get('use_ensemble', False)
    use_fs       = cfg.get('use_feature_selection', False)
    use_calib    = cfg.get('use_calibration', False)
    calib_method = cfg.get('calibration_method', 'sigmoid')

    logging.info("=" * 65)
    logging.info(f"Timeframe: {tf} | USD/CHF Pattern Model v1.0-fix")
    logging.info(f"  active={cfg.get('active_for_live',True)} | adx_min={cfg.get('adx_min')} | "
                 f"n_splits={n_splits} | ensemble={use_ensemble} | fs={use_fs} | calib={use_calib}({calib_method})")
    logging.info("=" * 65)

    # ── تحميل البيانات — الطريقة الجديدة أو القديمة ──────────
    SYMBOL_SAFE = "USDCHF"
    RAW_CSV_MAP = {
        "5min":  f"{SYMBOL_SAFE}_5min_raw.csv",
        "15min": f"{SYMBOL_SAFE}_15min_raw.csv",
        "45min": f"{SYMBOL_SAFE}_45min_raw.csv",
        "1h":    f"{SYMBOL_SAFE}_1h_raw.csv",
        "2h":    f"{SYMBOL_SAFE}_2h_raw.csv",
        "4h":    f"{SYMBOL_SAFE}_4h_raw.csv",
        "1d":    f"{SYMBOL_SAFE}_1d_raw.csv",
        "1W":    f"{SYMBOL_SAFE}_1W_raw.csv",
    }

    tf_csv = RAW_CSV_MAP.get(tf)

    # الأولوية: ملف مخصص للـ timeframe → resample من 5min
    if tf_csv and os.path.exists(tf_csv):
        logging.info(f"📂 تحميل مباشر من: {tf_csv}")
        df_tf = pd.read_csv(tf_csv)
        df_tf['datetime'] = pd.to_datetime(df_tf['datetime'])
        df_tf.sort_values('datetime', inplace=True)
        df_tf.reset_index(drop=True, inplace=True)
        # تأكد من وجود volume
        if 'volume' not in df_tf.columns:
            df_tf['volume'] = 1_000_000
        logging.info(f"  ✅ محمّل: {len(df_tf):,} شمعة")
    elif df_raw is not None:
        logging.info(f"⚙️  resample من 5min CSV (الطريقة القديمة) — يُنصح بتشغيل fetch_data_usdchf_multitf.py")
        df_tf = (
            df_raw.set_index('datetime')
            .resample(cfg['resample_rule'])
            .agg({'open':'first','high':'max','low':'min','close':'last','volume':'sum'})
            .dropna().reset_index()
        )
        if 'datetime' not in df_tf.columns:
            df_tf = df_tf.rename(columns={df_tf.columns[0]: 'datetime'})
        df_tf['datetime'] = pd.to_datetime(df_tf['datetime'])
    else:
        logging.error(f"❌ لا يوجد ملف {tf_csv} ولا df_raw — شغّل fetch_data_usdchf_multitf.py أولاً")
        return None

    # v4.7: قطع البيانات — نستخدم كل البيانات المتاحة لو كانت أقل من المطلوب
    if cfg.get('years'):
        requested_cut = df_tf['datetime'].max() - timedelta(days=365 * cfg['years'])
        actual_oldest = df_tf['datetime'].min()
        # استخدم الأحدث من: المطلوب أو المتاح
        cut   = max(requested_cut, actual_oldest)
        df_tf = df_tf[df_tf['datetime'] >= cut].copy()
        avail_years = (df_tf['datetime'].max() - df_tf['datetime'].min()).days / 365
        logging.info(f"  البيانات المتاحة: {avail_years:.1f} سنة (مطلوب: {cfg['years']} سنة)")

    logging.info(f"Candles: {len(df_tf):,} | {df_tf['datetime'].min().date()} → {df_tf['datetime'].max().date()}")

    if len(df_tf) < min_candles:
        logging.error(f"❌ Not enough data ({len(df_tf)} < {min_candles})")
        return None

    fe = PatternFeatureEngineerV4()
    fe._use_median_split = cfg.get('use_median_split', False)
    fe._use_directional  = cfg.get('use_directional',  False)  # v4.5
    df_feat = fe.create_features(df_tf, config=cfg, shift=True)
    df_tgt  = fe.create_target(df_feat, horizon=horizon, min_return=min_return, tf_name=tf)

    if df_tgt.empty or len(df_tgt) < max(min_candles // 3, 30):
        logging.error(f"❌ Not enough rows after target: {len(df_tgt)}")
        return None

    pos_ratio = df_tgt['target'].mean()
    if not (0.15 <= pos_ratio <= 0.85):
        logging.warning(f"Imbalanced ({pos_ratio:.1%}) — median fallback")
        df_tgt2 = df_feat.copy()
        df_tgt2['future_close'] = df_tgt2['close'].shift(-horizon)
        df_tgt2['pct_change']   = (df_tgt2['future_close'] - df_tgt2['close']) / (df_tgt2['close'] + 1e-9)
        df_tgt2['target']       = (df_tgt2['pct_change'] > df_tgt2['pct_change'].median()).astype(int)
        df_tgt2.dropna(subset=['future_close'], inplace=True)
        df_tgt = df_tgt2

    df_tgt = df_tgt.reset_index(drop=True)
    X = df_tgt[fe.feature_columns]
    y = df_tgt['target']
    logging.info(f"X.shape={X.shape} | positive={y.mean():.1%}")

    stats = {}
    for pc in ['hs_pattern','double_pattern','triple_pattern','triangle_pattern',
               'wedge_pattern','flag_pattern','harmonic_pattern','elliott_impulse',
               'pattern_cluster','strong_cluster']:
        if pc in df_tgt.columns:
            stats[pc] = int((df_tgt[pc] > 0).sum())
    logging.info(f"Pattern stats: {stats}")

    try:
        fold_res, best_model, best_fe = walk_forward_v4(
            X, y, df_tgt, cfg, n_splits=n_splits,
            purge_pct=0.005, tf_name=tf)
    except ValueError as e:
        logging.error(f"❌ Walk-Forward failed: {e}")
        return None

    if not fold_res:
        logging.error(f"❌ No valid folds for {tf}")
        return None

    wf_accs  = [f['accuracy']  for f in fold_res]
    wf_aucs  = [f['auc']       for f in fold_res]
    wf_prs   = [f['precision'] for f in fold_res]
    wf_f1s   = [f['f1']        for f in fold_res]
    wf_pfs   = [f['backtest']['profit_factor'] for f in fold_res
                if f.get('backtest') and f['backtest'].get('total_trades', 0) > 0]
    wf_wrs   = [f['backtest']['win_rate'] for f in fold_res
                if f.get('backtest') and f['backtest'].get('total_trades', 0) > 0]
    wf_trades = [f['backtest']['total_trades'] for f in fold_res
                 if f.get('backtest') and f['backtest'].get('total_trades', 0) > 0]
    wf_sharpes = [f['backtest'].get('sharpe_ratio', 0) for f in fold_res
                  if f.get('backtest') and f['backtest'].get('total_trades', 0) > 0]

    logging.info(f"  Avg Acc: {np.mean(wf_accs):.2%} | Avg AUC: {np.mean(wf_aucs):.3f} | Avg F1: {np.mean(wf_f1s):.3f}")
    if wf_pfs:
        logging.info(f"  Avg PF: {np.mean(wf_pfs):.2f} | WR: {np.mean(wf_wrs):.1f}% | Sharpe: {np.mean(wf_sharpes):.2f}")

    # ── موديل نهائي على كل البيانات ──────────────────────────
    fef_full = PatternFeatureEngineerV4()
    fef_full._use_median_split = cfg.get('use_median_split', False)
    fef_full._use_directional  = cfg.get('use_directional',  False)  # v4.5
    fef_full.fit_scaler(X)
    Xfs_full = fef_full.transform(X)

    if use_fs:
        fef_full.fit_feature_selection(Xfs_full, y)
        feat_idx = [fef_full.feature_columns.index(c) for c in fef_full.selected_cols
                    if c in fef_full.feature_columns]
        Xfs = Xfs_full[:, feat_idx]
    else:
        Xfs = Xfs_full

    neg_f   = (y == 0).sum(); pos_f = (y == 1).sum()
    spw_cap = cfg.get('spw_cap', 5.0)  # v4.3
    spw_f   = min(neg_f / pos_f, spw_cap) if pos_f > 0 else 1.0
    logging.info(f"  Final model: neg={neg_f} pos={pos_f} spw={spw_f:.2f}")

    if use_ensemble:
        final = build_ensemble_model(spw_f, tf)
        if use_calib:
            final = build_calibrated_model(final, method=calib_method)
        final.fit(Xfs, y)
    else:
        final = get_xgb_params_for_tf(tf, spw_f)
        if use_calib:
            final.set_params(early_stopping_rounds=None, eval_metric='logloss')
            final = build_calibrated_model(final, method=calib_method)
            final.fit(Xfs, y)
        else:
            final.set_params(early_stopping_rounds=None, eval_metric='logloss')
            final.fit(Xfs, y)

    # Feature importance
    feat_imp = pd.DataFrame({'feature': fef_full.feature_columns, 'importance': 0.0})
    try:
        if hasattr(final, 'feature_importances_'):
            feat_imp['importance'] = final.feature_importances_
        elif hasattr(final, 'estimator') and hasattr(final.estimator, 'feature_importances_'):
            feat_imp['importance'] = final.estimator.feature_importances_
        elif hasattr(final, 'named_estimators_'):
            xgb_est = final.named_estimators_.get('xgb')
            if xgb_est and hasattr(xgb_est, 'feature_importances_'):
                imp = xgb_est.feature_importances_
                if use_fs and fef_full.selected_cols:
                    full_imp = np.zeros(len(fef_full.feature_columns))
                    for idx, col in enumerate(fef_full.selected_cols):
                        if col in fef_full.feature_columns and idx < len(imp):
                            full_imp[fef_full.feature_columns.index(col)] = imp[idx]
                    feat_imp['importance'] = full_imp
                else:
                    feat_imp['importance'] = imp
    except Exception as e:
        logging.warning(f"Could not extract feature importance: {e}")
    feat_imp = feat_imp.sort_values('importance', ascending=False).head(25)

    # Holdout backtest
    hs = int(len(df_tgt) * 0.80)
    hmbt = BacktesterV4(
        min_pattern_quality=cfg.get('min_pattern_quality', 0.10),
        confidence_threshold=cfg.get('confidence_threshold', 0.55),
        adx_min=cfg.get('adx_min', 10.0),
        atr_sl_mult=cfg.get('atr_sl_mult', ATR_SL_MULT),
        atr_tp_mult=cfg.get('atr_tp_mult', ATR_TP_MULT),
        require_htf_align=cfg.get('require_htf_align', True),           # v4.4
        htf_override_threshold=cfg.get('htf_override_threshold', 0.68), # v4.4
    ).run(df_tgt.iloc[hs:].copy(), final, fef_full,
          fef_full.feature_columns, cfg, use_fold=False)

    if hmbt and hmbt.get('total_trades', 0) > 0:
        logging.info(
            f"✅ Holdout | T:{hmbt['total_trades']} WR:{hmbt['win_rate']}% "
            f"PF:{hmbt['profit_factor']} R:{hmbt['total_return_pct']}% "
            f"Sharpe:{hmbt.get('sharpe_ratio',0):.2f}"
        )

    # ── حفظ ──────────────────────────────────────────────────
    os.makedirs(MODELS_DIR, exist_ok=True)
    mpath = os.path.join(MODELS_DIR, f"{SYMBOL_SAFE}_{tf}_patterns_v4.pkl")
    joblib.dump({
        'model':              final,
        'scaler':             fef_full.scaler,
        'feature_columns':    fef_full.feature_columns,
        'selected_cols':      fef_full.selected_cols,
        'config':             cfg,
        'feature_importance': feat_imp.to_dict('records'),
        'version':            'v4.1-fast'
    }, mpath)

    consistency_pct = round(sum(1 for p in wf_pfs if p > 1) / max(len(wf_pfs), 1) * 100, 1) if wf_pfs else 0
    jpath = os.path.join(MODELS_DIR, f"{SYMBOL_SAFE}_{tf}_patterns_metrics_v4.json")
    with open(jpath, 'w', encoding='utf-8') as f:
        json.dump({
            'version':    'v4.0_usdchf_45min_fix',
            'symbol':     'USD/CHF',
            'timeframe':  tf,
            'active_for_live': cfg.get('active_for_live', True),
            'use_ensemble': use_ensemble,
            'use_feature_selection': use_fs,
            'use_calibration': use_calib,
            'walk_forward': {
                'n_folds':             len(fold_res),
                'avg_accuracy':        round(float(np.mean(wf_accs)),  4) if wf_accs else 0,
                'avg_auc':             round(float(np.mean(wf_aucs)),  4) if wf_aucs else 0,
                'avg_f1':              round(float(np.mean(wf_f1s)),   4) if wf_f1s  else 0,
                'avg_precision':       round(float(np.mean(wf_prs)),   4) if wf_prs  else 0,
                'avg_win_rate':        round(float(np.mean(wf_wrs)),   2) if wf_wrs  else 0,
                'avg_profit_factor':   round(float(np.mean(wf_pfs)),   2) if wf_pfs  else 0,
                'avg_sharpe':          round(float(np.mean(wf_sharpes)), 2) if wf_sharpes else 0,
                'avg_trades_per_fold': round(float(np.mean(wf_trades)), 0) if wf_trades else 0,
                'consistency_pct':     consistency_pct,
                'folds':               fold_res
            },
            'holdout_last_20pct': hmbt,
            'config':             cfg,
            'training_samples':   len(df_tgt),
            'features':           len(fef_full.feature_columns),
            'selected_features':  len(fef_full.selected_cols) if fef_full.selected_cols else len(fef_full.feature_columns),
            'feature_importance': feat_imp.to_dict('records'),
        }, f, indent=2, default=str, ensure_ascii=False)

    logging.info(f"💾 Saved: {mpath}")
    return final, fef_full


# ══════════════════════════════════════════════════════════════
# MULTI-TIMEFRAME DECISION v4
# ══════════════════════════════════════════════════════════════
def multi_timeframe_decision(signals: dict) -> dict:
    if not signals:
        return {"final_signal": "NO TRADE", "confidence": 0.0}

    TF_W = {"1d":4,"4h":3,"2h":2.5,"1h":2,"45min":1.5,"15min":1,"5min":0.5}

    dirs = {}; probs = {}; quals = {}; aucs = {}; sharpes = {}
    for tf, v in signals.items():
        if not TIMEFRAME_CONFIG.get(tf, {}).get('active_for_live', True):
            continue
        dirs[tf]    = v.get('direction', 0)
        probs[tf]   = v.get('prob', 0.5)
        quals[tf]   = v.get('pattern_quality', 0.0)
        aucs[tf]    = v.get('auc', 0.5)
        sharpes[tf] = max(v.get('sharpe', 0.0), 0.0)

    wd = tw = 0.0
    for tf, d in dirs.items():
        sharpe_boost = 1.0 + min(sharpes.get(tf, 0.0) * 0.1, 0.5)
        w = TF_W.get(tf, 1.0) * sharpe_boost
        wd += d * w * probs.get(tf, 0.5)
        tw += w
    if tw > 0:
        wd /= tw

    avg_q    = float(np.mean(list(quals.values()))) if quals else 0.0
    avg_p    = float(np.mean(list(probs.values()))) if probs else 0.5
    avg_auc  = float(np.mean(list(aucs.values())))  if aucs  else 0.5
    avg_shrp = float(np.mean(list(sharpes.values()))) if sharpes else 0.0

    sig = "BUY" if wd > 0.25 else ("SELL" if wd < -0.25 else "NO TRADE")

    return {
        "final_signal":       sig,
        "confidence":         round(avg_p, 3),
        "weighted_direction": round(wd, 3),
        "avg_quality":        round(avg_q, 3),
        "avg_auc":            round(avg_auc, 3),
        "avg_sharpe":         round(avg_shrp, 3),
        "tf_signals":         signals
    }


# ══════════════════════════════════════════════════════════════
# TRAIN ALL
# ══════════════════════════════════════════════════════════════
def train_all(csv_path="/content/USDCHF_5min_raw.csv"):
    logging.info("=" * 70)
    logging.info("train_patterns_usdchf_v1.py  —  USD/CHF v4.8 Best-of-All Configs")
    logging.info("Fixes vs v4.1-fast:")
    logging.info("  - 1h: horizon=8, buy_thr=0.57, adx_min=14, pq=0.15, sigmoid calib")
    logging.info("  - 4h: horizon=5, buy_thr=0.55, adx_min=12, pq=0.10, sigmoid calib")
    logging.info("  - 1d: horizon=12, use_calibration=False (isotonic was broken)")
    logging.info("  - Multi-TF CSVs: كل timeframe يُحمَّل من ملفه المخصص")
    logging.info("  - create_target: better balance for 1h/4h (quantile=0.55)")
    logging.info(f"SL={ATR_SL_MULT}×ATR | Spread={SPREAD_BPS}bps")
    logging.info("=" * 70)

    # ── تحميل ذكي: كل TF يحمّل من ملفه إن وُجد ──────────────
    RAW_CSV_MAP = {
        "5min":  f"{SYMBOL_SAFE}_5min_raw.csv",
        "15min": f"{SYMBOL_SAFE}_15min_raw.csv",
        "45min": f"{SYMBOL_SAFE}_45min_raw.csv",
        "1h":    f"{SYMBOL_SAFE}_1h_raw.csv",
        "2h":    f"{SYMBOL_SAFE}_2h_raw.csv",
        "4h":    f"{SYMBOL_SAFE}_4h_raw.csv",
        "1d":    f"{SYMBOL_SAFE}_1d_raw.csv",
        "1W":    f"{SYMBOL_SAFE}_1W_raw.csv",
    }

    # نجيب الـ 5min fallback لو محتاجينه
    df_raw_5min = None
    if os.path.exists(csv_path):
        df_raw_5min = pd.read_csv(csv_path)
        df_raw_5min['datetime'] = pd.to_datetime(df_raw_5min['datetime'])
        df_raw_5min.sort_values('datetime', inplace=True)
        df_raw_5min.reset_index(drop=True, inplace=True)
        logging.info(f"5min CSV: {len(df_raw_5min):,} شمعة | "
                     f"{df_raw_5min['datetime'].min().date()} → {df_raw_5min['datetime'].max().date()}")
    else:
        logging.warning(f"⚠️ {csv_path} غير موجود — سيُستخدم الملف المخصص لكل TF فقط")

    # عرض حالة الملفات
    logging.info("\n📊 حالة ملفات البيانات:")
    for tf, tf_csv in RAW_CSV_MAP.items():
        if os.path.exists(tf_csv):
            df_tmp = pd.read_csv(tf_csv, nrows=1)
            try:
                nrows = sum(1 for _ in open(tf_csv)) - 1
                logging.info(f"  ✅ {tf:6s}: {tf_csv} ({nrows:,} شمعة)")
            except Exception:
                logging.info(f"  ✅ {tf:6s}: {tf_csv}")
        else:
            src = "→ سيُستخدم 5min resample" if df_raw_5min is not None else "→ ❌ غير متاح"
            logging.info(f"  ⚠️  {tf:6s}: {tf_csv} غير موجود {src}")

    results    = {}
    wf_summary = {}

    for tf in TIMEFRAME_CONFIG:
        active = TIMEFRAME_CONFIG[tf].get('active_for_live', True)

        if SKIP_INACTIVE_TF and not active:
            logging.info(f"\n⏭️  Skipping {tf} (active_for_live=False, SKIP_INACTIVE_TF=True)")
            results[tf] = "⏭️ Skipped (HTF only)"
            continue

        logging.info(f"\n{'─'*50}\nTimeframe: {tf} | active={active}")

        # اختر مصدر البيانات: ملف مخصص أو fallback 5min
        tf_csv  = RAW_CSV_MAP.get(tf)
        use_raw = df_raw_5min if (not tf_csv or not os.path.exists(tf_csv)) else None

        try:
            res = train_model(use_raw, tf)
            results[tf] = "✅ OK" if res else "❌ Failed"
        except Exception as e:
            logging.error(f"❌ Error in {tf}: {e}", exc_info=True)
            results[tf] = f"❌ Error: {e}"

        jpath = os.path.join(MODELS_DIR, f"{SYMBOL_SAFE}_{tf}_patterns_metrics_v4.json")
        if os.path.exists(jpath):
            with open(jpath, encoding='utf-8') as f:
                m  = json.load(f)
                wf = m.get('walk_forward', {})
                wf_summary[tf] = {
                    'active':  '✅' if m.get('active_for_live', True) else '⛔',
                    'ens':     '🔀' if m.get('use_ensemble') else '—',
                    'folds':   wf.get('n_folds', '-'),
                    'acc':     wf.get('avg_accuracy', '-'),
                    'auc':     wf.get('avg_auc', '-'),
                    'pf':      wf.get('avg_profit_factor', '-'),
                    'wr':      wf.get('avg_win_rate', '-'),
                    'sharpe':  wf.get('avg_sharpe', '-'),
                    'consist': wf.get('consistency_pct', '-'),
                }

    logging.info("\n" + "=" * 70)
    logging.info("Final Summary — USD/CHF Pattern Models v4.1-fast:")
    logging.info(f"  {'TF':<8} {'Live':<5} {'Ens':<4} {'Folds':<6} {'Acc':<8} {'AUC':<7} {'WR%':<7} {'PF':<6} {'Sharpe':<8} {'Consist%'}")
    for tf in TIMEFRAME_CONFIG:
        wf = wf_summary.get(tf, {})
        status = results.get(tf, '-')
        logging.info(
            f"  {tf:<8} {str(wf.get('active','-')):<5} {str(wf.get('ens','-')):<4} "
            f"{str(wf.get('folds','-')):<6} {str(wf.get('acc','-')):<8} "
            f"{str(wf.get('auc','-')):<7} {str(wf.get('wr','-')):<7} "
            f"{str(wf.get('pf','-')):<6} {str(wf.get('sharpe','-')):<8} "
            f"{str(wf.get('consist','-'))}%  {status}"
        )
    logging.info(f"\nAll files saved to: {MODELS_DIR}/")
    logging.info("Tip: Set SKIP_INACTIVE_TF=True to skip 1W (HTF context only)")
    logging.info("=" * 70)


# ══════════════════════════════════════════════════════════════
if __name__ == "__main__":

    csv_path = "/content/USDCHF_5min_raw.csv"
    train_all(csv_path=csv_path)

In [11]:
# ============================================================
# show_results_usdchf.py  —  عرض نتائج نماذج USD/CHF
# ============================================================

import os
import json
import pandas as pd
from pathlib import Path

MODELS_DIR = "/content/models/usdchf_patterns_v4"
SYMBOL = "USDCHF"

TIMEFRAMES = ["5min", "15min", "45min", "1h", "2h", "4h", "1d", "1W"]

def load_metrics(tf: str) -> dict | None:
    path = os.path.join(MODELS_DIR, f"{SYMBOL}_{tf}_patterns_metrics_v4.json")
    if not os.path.exists(path):
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def format_pct(val, decimals=1):
    if val is None or val == "-":
        return "-"
    try:
        return f"{float(val):.{decimals}f}%"
    except:
        return str(val)

def format_num(val, decimals=2):
    if val is None or val == "-":
        return "-"
    try:
        return f"{float(val):.{decimals}f}"
    except:
        return str(val)

def main():
    print("=" * 90)
    print("  USD/CHF Pattern Models — Results Summary")
    print("=" * 90)

    rows = []
    details = {}

    for tf in TIMEFRAMES:
        m = load_metrics(tf)
        if m is None:
            rows.append({
                "TF": tf,
                "Live": "❌",
                "Folds": "-",
                "Acc": "-",
                "AUC": "-",
                "WR%": "-",
                "PF": "-",
                "Sharpe": "-",
                "Consist%": "-",
                "Holdout WR": "-",
                "Holdout PF": "-",
                "Trades": "-",
            })
            continue

        wf = m.get("walk_forward", {})
        ho = m.get("holdout_last_20pct", {}) or {}

        live = "✅" if m.get("active_for_live", True) else "⛔"
        folds = wf.get("n_folds", "-")
        acc = format_num(wf.get("avg_accuracy"), 3)
        auc = format_num(wf.get("avg_auc"), 3)
        wr = format_num(wf.get("avg_win_rate"), 1)
        pf = format_num(wf.get("avg_profit_factor"), 2)
        sharpe = format_num(wf.get("avg_sharpe"), 2)
        consist = format_num(wf.get("consistency_pct"), 0)
        ho_wr = format_num(ho.get("win_rate"), 1) if ho else "-"
        ho_pf = format_num(ho.get("profit_factor"), 2) if ho else "-"
        trades = ho.get("total_trades", "-") if ho else "-"

        rows.append({
            "TF": tf,
            "Live": live,
            "Folds": folds,
            "Acc": acc,
            "AUC": auc,
            "WR%": wr,
            "PF": pf,
            "Sharpe": sharpe,
            "Consist%": consist,
            "Holdout WR": ho_wr,
            "Holdout PF": ho_pf,
            "Trades": trades,
        })

        details[tf] = m

    df = pd.DataFrame(rows)
    print("\n📊 Walk-Forward + Holdout Summary:\n")
    print(df.to_string(index=False))

    # ── تفاصيل إضافية لكل فريم ──
    print("\n" + "=" * 90)
    print("  Detailed per Timeframe")
    print("=" * 90)

    for tf in TIMEFRAMES:
        m = details.get(tf)
        if not m:
            print(f"\n[{tf}] — لا يوجد ملف نتائج")
            continue

        wf = m.get("walk_forward", {})
        ho = m.get("holdout_last_20pct", {}) or {}
        cfg = m.get("config", {})

        print(f"\n{'─'*70}")
        print(f"  {tf.upper()}  |  active={m.get('active_for_live')}  |  samples={m.get('training_samples')}  |  features={m.get('selected_features')}/{m.get('features')}")
        print(f"{'─'*70}")

        print(f"  Walk-Forward ({wf.get('n_folds')} folds):")
        print(f"    Acc={wf.get('avg_accuracy')}  AUC={wf.get('avg_auc')}  F1={wf.get('avg_f1')}")
        print(f"    WR={wf.get('avg_win_rate')}%  PF={wf.get('avg_profit_factor')}  Sharpe={wf.get('avg_sharpe')}  Consist={wf.get('consistency_pct')}%")
        print(f"    Avg trades/fold ≈ {wf.get('avg_trades_per_fold')}")

        if ho and ho.get("total_trades", 0) > 0:
            print(f"  Holdout (last 20%):")
            print(f"    Trades={ho.get('total_trades')}  WR={ho.get('win_rate')}%  PF={ho.get('profit_factor')}")
            print(f"    Return={ho.get('total_return_pct')}%  MaxDD={ho.get('max_drawdown_pct')}%  Sharpe={ho.get('sharpe_ratio')}")
            print(f"    Expectancy={ho.get('expectancy')}")
        else:
            print("  Holdout: لا توجد صفقات أو فشل الـ backtest")

        # أهم الـ features
        fi = m.get("feature_importance", [])
        if fi:
            print("  Top 8 Features:")
            for item in fi[:8]:
                print(f"    {item['feature']:<35} {item['importance']:.4f}")

        # إعدادات مهمة
        print(f"  Config highlights:")
        print(f"    horizon={cfg.get('horizon')}  pq_min={cfg.get('min_pattern_quality')}  "
              f"buy_thr={cfg.get('buy_threshold')}  sell_thr={cfg.get('sell_threshold')}")
        print(f"    SL={cfg.get('atr_sl_mult')}×ATR  TP={cfg.get('atr_tp_mult')}×ATR  "
              f"ADX_min={cfg.get('adx_min')}  directional={cfg.get('use_directional')}")

    print("\n" + "=" * 90)
    print("Done.")
    print("=" * 90)


if __name__ == "__main__":
    main()

  USD/CHF Pattern Models — Results Summary

📊 Walk-Forward + Holdout Summary:

   TF Live  Folds   Acc   AUC  WR%   PF Sharpe Consist% Holdout WR Holdout PF  Trades
 5min    ✅      6 0.605 0.673 64.2 2.39   0.07      100       68.5       3.24     400
15min    ✅      6 0.583 0.629 42.9 1.47   0.14      100       52.5       2.39     680
45min    ✅      5 0.529 0.536 43.9 1.87   0.17      100       50.5       2.68     321
   1h    ✅      5 0.514 0.530 35.3 1.40   0.10       67       59.0       3.54     229
   2h    ✅      5 0.511 0.534 34.5 1.55   0.16      100       52.0       3.33     123
   4h    ✅      4 0.565 0.562 33.3 1.78   0.05      100       60.2       4.59      88
   1d    ✅      3 0.542 0.559 24.9 1.23   0.18       50       49.2       2.77      63
   1W    ✅      3 0.532 0.554 37.6 1.58  -0.11      100       40.0       1.81      15

  Detailed per Timeframe

──────────────────────────────────────────────────────────────────────
  5MIN  |  active=True  |  samples=465663  |  fea

## — Download Models

In [12]:
# ============================================================
# download_models


import os
import shutil
from google.colab import files

MODELS_DIR = "/content/models/usdchf_patterns_v4"
ZIP_NAME   = "usdchf_classic_models.zip"

# ── تحقق من وجود الفولدر ──
if not os.path.exists(MODELS_DIR):
    print(f"❌ الفولدر '{MODELS_DIR}' مش موجود!")
    print("تأكد إن usdchf_patterns_v35.py اشتغل وحفظ الملفات أولاً.")
else:
    # ── اعرض الملفات الموجودة ──
    all_files = os.listdir(MODELS_DIR)
    pkl_files  = [f for f in all_files if f.endswith('.pkl')]
    json_files = [f for f in all_files if f.endswith('.json')]

    print(f"📁 محتوى '{MODELS_DIR}':")
    print(f"   .pkl  (موديلات): {len(pkl_files)}")
    print(f"   .json (مقاييس):  {len(json_files)}")
    print()
    for f in sorted(all_files):
        size = os.path.getsize(os.path.join(MODELS_DIR, f))
        print(f"   {f:<45} {size/1024:.1f} KB")

    # ── اضغط الفولدر ──
    print(f"\n📦 جاري ضغط الفولدر ...")
    shutil.make_archive(
        ZIP_NAME.replace('.zip', ''),  # اسم الملف
        'zip',                          # نوع الضغط
        '.',                            # المجلد الأساسي
        MODELS_DIR                      # الفولدر المراد ضغطه
    )

    size_mb = os.path.getsize(ZIP_NAME) / (1024 * 1024)
    print(f"✅ تم إنشاء: {ZIP_NAME} ({size_mb:.1f} MB)")

    # ── نزّل الملف ──
    print(f"⬇️  جاري التحميل ...")
    files.download(ZIP_NAME)
    print(f"✅ تم! افتح الـ ZIP وهتلاقي فولدر 'models/patterns' بداخله.")

📁 محتوى '/content/models/usdchf_patterns_v4':
   .pkl  (موديلات): 8
   .json (مقاييس):  8

   USDCHF_15min_patterns_metrics_v4.json         510.5 KB
   USDCHF_15min_patterns_v4.pkl                  1049.6 KB
   USDCHF_1W_patterns_metrics_v4.json            14.4 KB
   USDCHF_1W_patterns_v4.pkl                     433.6 KB
   USDCHF_1d_patterns_metrics_v4.json            32.8 KB
   USDCHF_1d_patterns_v4.pkl                     449.1 KB
   USDCHF_1h_patterns_metrics_v4.json            87.6 KB
   USDCHF_1h_patterns_v4.pkl                     360.1 KB
   USDCHF_2h_patterns_metrics_v4.json            65.1 KB
   USDCHF_2h_patterns_v4.pkl                     353.4 KB
   USDCHF_45min_patterns_metrics_v4.json         109.4 KB
   USDCHF_45min_patterns_v4.pkl                  361.4 KB
   USDCHF_4h_patterns_metrics_v4.json            24.8 KB
   USDCHF_4h_patterns_v4.pkl                     349.5 KB
   USDCHF_5min_patterns_metrics_v4.json          370.1 KB
   USDCHF_5min_patterns_v4.pkl             

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ تم! افتح الـ ZIP وهتلاقي فولدر 'models/patterns' بداخله.
